# A zoo of phase-dependent functions
Plots and cross-checks the scalar functions behind the analytic (PRR 8) closed form for a 4×2 array with unequal tweezer phases (`PHASE_DEPENDENT_toy_params`, φₙ = 0°, 30°, 60°, 90°): the optical susceptibilities χ_c(ω), χ_c*(−ω), their difference η_c and product ξ_c; the mechanical susceptibilities; the collective functions 𝒟₀ and 𝒟ₙ; the phase-dependent 𝒫 = 𝒟₀² − 𝒟₂𝒟₋₂ and its per-dimension and per-pair breakdowns; and the related 𝒦_j, ℋ_j, 𝒰_j. Several cells verify that different definitions agree.

Runtime: about 5 min.

# START

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

set_plt_rcParams_to_xmgrace_style()

ω_range = np.arange(279, 321 + 0.01, 0.01) * kHz
full_ω_range = np.arange(-1_000, 1_000 + 0.5, 0.5) * kHz

num_traps, Cartesian_axes_per_trap = 4, 2
mtrap = MultiTrap(
    num_traps=num_traps,
    Cartesian_axes_per_trap=Cartesian_axes_per_trap,
    ω_range=ω_range,
    cavity_kwargs=DEFAULT_cavity_kwargs,
    gas_kwargs=DEFAULT_gas_kwargs,
    toy_params=PHASE_DEPENDENT_toy_params,
    tweezer_phase_sign_convention=-1,
)

def Pjk_str(dim_pair):
    jn, km = dim_pair[0].label, dim_pair[1].label
    j, n = jn[0], jn[1:]
    k, m = km[0], km[1:]
    jn_string = f'{j}_{n}'
    km_string = f'{k}_{m}'
    return rf'$\mathcal{{P}}_{{{jn_string + km_string}}}$'

def Pj_str(dim):
    j, n = dim.label[0], dim.label[1:]
    jn_string = f'{j}_{n}'
    return rf'$\mathcal{{P}}_{{{jn_string}}}$'

def Pj_str_1(dim):
    j, n = dim.label[0], dim.label[1:]
    jn_string = f'{j}_{n}'
    return rf'$\mathcal{{P}}^{{(1)}}_{{{jn_string}}}$'

def Pj_str_2(dim):
    j, n = dim.label[0], dim.label[1:]
    jn_string = f'{j}_{n}'
    return rf'$\mathcal{{P}}^{{(2)}}_{{{jn_string}}}$'

def Ωj_str(dim):
    j, n = dim.label[0], dim.label[1:]
    jn_string = f'{j}_{n}'
    return rf'$\Omega_{{{jn_string}}}$'

def Kj_str(dim):
    j, n = dim.label[0], dim.label[1:]
    jn_string = f'{j}_{n}'
    return rf'$\mathcal{{K}}_{{{jn_string}}}$'

def Hj_str(dim):
    j, n = dim.label[0], dim.label[1:]
    jn_string = f'{j}_{n}'
    return rf'$\mathcal{{H}}_{{{jn_string}}}$'

def Hj_dag_str(dim):
    j, n = dim.label[0], dim.label[1:]
    jn_string = f'{j}_{n}'
    return rf'$\mathcal{{H}}_{{{jn_string}}}^\dagger$'

def Uj_str(dim):
    j, n = dim.label[0], dim.label[1:]
    jn_string = f'{j}_{n}'
    return rf'$\mathcal{{U}}_{{{jn_string}}}$'

def Uj_dag_str(dim):
    j, n = dim.label[0], dim.label[1:]
    jn_string = f'{j}_{n}'
    return rf'$\mathcal{{U}}_{{{jn_string}}}^\dagger$'

def Nj_str(dim):
    j, n = dim.label[0], dim.label[1:]
    jn_string = f'{j}_{n}'
    return rf'$\mathcal{{N}}_{{{jn_string}}}$'

option_to_latex_M = {
    'K': r'$1 + \sum_j g_j^2 \mu_j (\eta_{\mathrm{c}} - \xi_{\mathrm{c}} \mathcal{K}_j)$',
    'H': r'$1 + \sum_j g_j^2 \mu_j (\eta_{\mathrm{c}} + \chi_{\mathrm{c}}(\omega) \mathcal{H}_j)$',
    'U': r'$1 + \sum_j g_j^2 \mu_j (\chi_{\mathrm{c}}(\omega) \mathcal{U}_j - \chi_{\mathrm{c}}^*(-\omega))$',
    'H_dag': r'$1 + \sum_j g_j^2 \mu_j (\eta_{\mathrm{c}} - \chi_{\mathrm{c}}^*(-\omega) \mathcal{H}_j^\dagger)$',
    'U_dag': r'$1 + \sum_j g_j^2 \mu_j (\chi_{\mathrm{c}}(\omega) - \chi_{\mathrm{c}}^*(-\omega)\mathcal{U}_j^\dagger)$',
}

option_to_latex_Nj = {
    'K': r'$\eta_{\mathrm{c}} - \xi_{\mathrm{c}} \mathcal{K}_j$',
    'H': r'$\eta_{\mathrm{c}} + \chi_{\mathrm{c}}(\omega) \mathcal{H}_j$',
    'U': r'$\chi_{\mathrm{c}}(\omega) \mathcal{U}_j - \chi_{\mathrm{c}}^*(-\omega)$',
    'H_dag': r'$\eta_{\mathrm{c}} - \chi_{\mathrm{c}}^*(-\omega) \mathcal{H}_j^\dagger$',
    'U_dag': r'$\chi_{\mathrm{c}}(\omega) - \chi_{\mathrm{c}}^*(-\omega)\mathcal{U}_j^\dagger$',
}

option_defs = {
    'K': 'i)',
    'H': 'ii)',
    'U': 'ii)',
    'H_dag': 'iv)',
    'U_dag': 'v)',
}

dim_to_colour = {
    dim: colour
    for (dim, colour) in zip(mtrap.dimensions.values(), all_colours_dict.values())
}

# latex_ω_units = {
#     2: r' [($2\pi\cdot$Hz)$^{\mathregular{2}}$]',
#     1: r' [$2\pi\cdot$Hz]',
#     -1: r' [($2\pi\cdot$Hz)$^{\mathregular{-1}}$]',
#     -2: r' [($2\pi\cdot$Hz)$^{\mathregular{-2}}$]',
# }


# Physical definitions

---

Optical susceptibility of photon annihilation $\chi_\mathrm{c}(\omega)$ and creation $\chi_\mathrm{c}^*(-\omega)$
- Units: [$\chi_\mathrm{c}(\omega)$] = ($2\pi \cdot$ Hz)$^{-1}$
- Hermitian conjugate: $[\chi_\mathrm{c}(\omega)]^\dagger = \chi_\mathrm{c}^*(-\omega)$

$$ \chi_\mathrm{c}(\omega) \equiv \left[ -i(\omega + \Delta) + \tfrac{\kappa}{2} \right]^{-1} $$
$$ \chi_\mathrm{c}^*(-\omega) = \left[ -i(\omega - \Delta) + \tfrac{\kappa}{2} \right]^{-1} $$

---

Hermitian product of optical susceptibilities $\xi_\mathrm{c}$
- Units: [$\xi_\mathrm{c}$] = ($2\pi \cdot$ Hz)$^{-2}$
- Hermitian: $\xi_\mathrm{c}^\dagger = \xi_\mathrm{c}$

$$
    \xi_\mathrm{c}
    \equiv
    \chi_\mathrm{c}(\omega) \chi_\mathrm{c}^*(-\omega)
    =
    \frac{1}{\Delta^2 - \omega^2 +\frac{\kappa^2}{4} -i \kappa \omega}
$$

---

Hermitian difference of optical susceptibilities
- Units: [$\eta_\mathrm{c}$] = ($2\pi \cdot$ Hz)$^{-1}$
- Anti-Hermitian: $\eta_\mathrm{c}^\dagger = -\eta_\mathrm{c}$
- $\eta_\mathrm{c} = 2i\Delta \xi_\mathrm{c}$
$$
    \eta_\mathrm{c}
    \equiv
    \chi_\mathrm{c}(\omega) - \chi_\mathrm{c}^*(-\omega) 
    =
    \frac{2i\Delta}{\Delta^2 - \omega^2 +\frac{\kappa^2}{4} -i \kappa \omega}
$$

---

Mechanical susceptibility of phonon annihilation $\chi_j(\omega)$ and creation $\chi_j^*(-\omega)$ of $j^\mathrm{th}$ mode
- Units: [$\chi_j(\omega)$] = ($2\pi \cdot$ Hz)$^{-1}$
- Hermitian conjugate: $[\chi_j(\omega)]^\dagger = \chi_j^*(-\omega)$
$$ \chi_j(\omega) \equiv \left[ -i(\omega - \Omega_j) + \tfrac{\Gamma_j}{2} \right]^{-1} $$
$$ \chi_j^*(-\omega) = \left[ -i(\omega + \Omega_j) + \tfrac{\Gamma_j}{2} \right]^{-1} $$

---

Hermitian difference of mechanical susceptibilities
- Units: [$\mu_j$] = ($2\pi \cdot$ Hz)$^{-1}$
- Anti-Hermitian: $\mu_j^\dagger = -\mu_j$
$$
    \mu_j
    \equiv
    \chi_j(\omega) - \chi_j^*(-\omega) = 2i\Omega_j \chi_j(\omega) \chi_j^*(-\omega)
    =
    \frac{-2i\Omega_j}{\Omega_j^2 - \omega^2 +\frac{\Gamma_j^2}{4} -i \Gamma_j \omega}
$$

---

Phase-dependent sum of square of photon-phonon coupling rates multiplied by Hermitian difference of mechanical susceptibility
- Units: [$\mathcal{D}_n$] = $2\pi \cdot$ Hz
- Hermitian conjugate: $\mathcal{D}_n^\dagger = -\mathcal{D}_{-n}$
$$ \mathcal{D}_n \equiv \sum_j g_j^2 \mu_j e^{i n \phi_j} $$

The 3 versions that appear are
$$ \mathcal{D}_0 = \sum_j g_j^2 \mu_j $$
$$ \mathcal{D}_2 = \sum_j g_j^2 \mu_j e^{2i  \phi_j}$$
$$ \mathcal{D}_{-2} = \sum_j g_j^2 \mu_j e^{-2i  \phi_j}$$

---

In fact, they always appear in the combination
$$ \mathcal{P} \equiv \mathcal{D}_0^2 - \mathcal{D}_2 \mathcal{D}_{-2} $$
- Units: [$\mathcal{P}$] = ($2\pi \cdot$ Hz)$^2$
- Hermitian: $\mathcal{P}^\dagger = \mathcal{P}$

where
$$
    \mathcal{D}_0^2
    =
    \left( \sum_j g_j^2 \mu_j \right) \left( \sum_j g_k^2 \mu_k \right)
    =
    \sum_j g_j^4 \mu_j^2 + \sum_j \sum_{k \neq j} g_j^2 g_k^2 \mu_j \mu_k
$$

and
$$
    \mathcal{D}_2 \mathcal{D}_{-2}
    =
    \left( \sum_j g_j^2 \mu_j e^{2i  \phi_j} \right) \left( \sum_j g_k^2 \mu_k e^{-2i  \phi_k} \right)
    \\
    \mathcal{D}_2 \mathcal{D}_{-2}
    =
    \sum_j g_j^4 \mu_j^2 + \frac{1}{2} \sum_j \sum_{k \neq j} g_j^2 g_k^2 \mu_j \mu_k \left[ e^{2i (\phi_j - \phi_k)} + e^{-2i (\phi_j - \phi_k)} \right]
    \\
    \mathcal{D}_2 \mathcal{D}_{-2}
    =
    \sum_j g_j^4 \mu_j^2 + \sum_j \sum_{k \neq j} g_j^2 g_k^2 \mu_j \mu_k \cos(2(\phi_j - \phi_k))
$$

hence the "diagonal" $k = j$ parts cancel leaving

$$
    \mathcal{P}
    =
    \sum_j \sum_{k \neq j} g_j^2 \mu_j g_k^2 \mu_k \left[ 1 - \cos(2(\phi_j - \phi_k)) \right]
    =
    \sum_j \sum_{k \neq j} g_j^2 \mu_j g_k^2 \mu_k \left[2 \sin^2 (\phi_j - \phi_k) \right]
$$

---

---

# GOAL: Verify all definitions of normalisation function $\mathcal{M}$ are equal

---

Explicit definition
- Units: [$\mathcal{M}$] = 1
- Hermitian: $\mathcal{M}^\dagger = \mathcal{M}$
$$
    \mathcal{M}
    \equiv
    1
    +
    \xi_\mathrm{c}
    \sum _j g_j^{2} \mu_j
    \left[
        2i\Delta
        -
        \sum_{k\neq j} g_{k}^{2} \mu _k \sin^2( \phi_j - \phi_k)
    \right]
$$

$$
    \mathcal{M}
    \equiv
    1
    +
    2i\xi_\mathrm{c}
    \sum _j g_j^{2} \mu_j
    \left[
        \Delta
        + i\mathcal{K}_j / 2
    \right]
$$

---

Definition with $\mathcal{P} \equiv \mathcal{D}_0^2 - \mathcal{D}_2 \mathcal{D}_{-2} = 2\sum_j g_j^2 \mu_j \sum_{k \neq j} g_k^2 \mu_k \sin^2(\phi_j - \phi_k)$
- Units: [$\mathcal{P}$] = ($2\pi \cdot$ Hz)$^2$
- Hermitian: $\mathcal{P}^\dagger = \mathcal{P}$
- $\mathcal{P} = 0$ when all $\phi_j$ are equal
- $\mathcal{P} = \sum_j g_j^2 \mu_j \mathcal{K}_j$
$$ \mathcal{M} \equiv 1 + \eta_\mathrm{c} \mathcal{D}_0 - \xi_\mathrm{c} \mathcal{P} $$

---

Definition with $\mathcal{K}_j \equiv 2\sum_{k \neq j} g_k^2 \mu_k \sin^2 (\phi_j - \phi_k) $ or $ \mathcal{K}_j \equiv \sum_k g_k^2 \mu_k \left( 1 - e^{2i(\phi_j - \phi_k)} \right) $
- Units: [$\mathcal{K}_j$] = $2\pi \cdot$ Hz
- Anti-Hermitian: $\mathcal{K}_j^\dagger = -\mathcal{K}_j$
- $\mathcal{K}_j = 0$ when all $\phi_j$ are equal
$$ \mathcal{M} \equiv 1 + \sum_j g_j^2 \mu_j (\eta_{\mathrm{c}} - \xi_{\mathrm{c}} \mathcal{K}_j) $$

---

Definition with $\mathcal{H}_j \equiv -\chi_\mathrm{c}^*(-\omega) \mathcal{K}_j$
- Units: [$\mathcal{H}_j$] = 1
- Hermitian conjugate: $\mathcal{H}_j^\dagger = - \frac{\chi_\mathrm{c}(\omega)}{\chi_\mathrm{c}^*(-\omega)} \mathcal{H}_j$
- $\mathcal{H}_j = 0$ when all $\phi_j$ are equal
- $\mathcal{H}_j = -\chi_\mathrm{c}^*(-\omega) \mathcal{K}_j$
$$ \mathcal{M} \equiv 1 + \sum_j g_j^2 \mu_j (\eta_{\mathrm{c}} + \chi_{\mathrm{c}}(\omega) \mathcal{H}_j) $$

Definition with $\mathcal{H}_j^\dagger \equiv \chi_\mathrm{c}(\omega) \mathcal{K}_j$
$$ \mathcal{M} \equiv 1 + \sum_j g_j^2 \mu_j (\eta_{\mathrm{c}} - \chi_{\mathrm{c}}^*(-\omega) \mathcal{H}_j^{\dagger}) $$

---

Definition with $\mathcal{U}_j \equiv 1 + \mathcal{H}_j$
- Units: [$\mathcal{U}_j$] = 1
- Hermitian conjugate: $\mathcal{U}_j^\dagger = 1 + \frac{\chi_\mathrm{c}(\omega)}{\chi_\mathrm{c}^*(-\omega)}( 1 - \mathcal{U}_j)$
- $\mathcal{U}_j = 1$ when all $\phi_j$ are equal
$$ \mathcal{M} \equiv 1 + \sum_j g_j^2 \mu_j (\chi_{\mathrm{c}}(\omega) \mathcal{U}_j - \chi_{\mathrm{c}}^*(-\omega)) $$

Definition with $\mathcal{U}_j^\dagger \equiv 1 + \mathcal{H}_j^\dagger$
$$ \mathcal{M} \equiv 1 + \sum_j g_j^2 \mu_j (\chi_{\mathrm{c}}(\omega) - \chi_{\mathrm{c}}^*(-\omega)\mathcal{U}_j^{\dagger}) $$

---


# QUESTION: why are there two different definitions of $\mathcal{K}_j$ that both work for $\mathcal{M}$ but are not equal?

# PART 1a: Optical susceptibility functions

## Optical photon susceptibilities $\chi_\mathrm{c}(\omega), \chi_\mathrm{c}^*(-\omega)$
- Units: [$\chi_\mathrm{c}(\omega)$] = ($2\pi \cdot$ Hz)$^{-1}$

Notes:
- Both are circles in the complex plane
- Both remain +Re
- Hermitian conjugation changes orientation / direction of the loop

In [ ]:
mosaic = [['1'], ['2'], ['3']]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

vals_χC = [mtrap.cavity.χC(ω) for ω in full_ω_range]
vals_χC_star = [np.conjugate(mtrap.cavity.χC(-ω)) for ω in full_ω_range]
Δ = mtrap.cavity.Δ

ax = ax_dict['1']
ax.set_title(r'$\chi_\mathrm{c}(\omega) \equiv [i(\omega + \Delta) + \frac{\kappa}{2}]^{-1}$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\chi_\mathrm{c}(\omega)$' + latex_ω_units[-1])
ax.set_xlim(-950, 950)
ax.set_ylim(-5e-7, 1e-6)
ax.hlines([0], -1e3, 1e3, color=black, alpha=0.1)
ax.vlines([0], -6e-7, 11e-7, color=black, alpha=0.1)
ax.vlines(-Δ / kHz, -6e-7, 11e-7, color=blue, alpha=0.2, ls='--', label=r'$-\Delta$')
ax.plot(full_ω_range / kHz, np.abs(vals_χC), label='Abs', lw=2, alpha=0.7, color=green)
ax.plot(full_ω_range / kHz, np.real(vals_χC), label='Re', lw=2, alpha=0.7, color=blue)
ax.plot(full_ω_range / kHz, np.imag(vals_χC), label='Im', lw=2, alpha=0.7, color=orange)
ax.legend()

ax = ax_dict['2']
ax.set_title(r'$\chi_\mathrm{c}^*(-\omega) \equiv [i(\omega - \Delta) + \frac{\kappa}{2}]^{-1}$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\chi_\mathrm{c}^*(-\omega)$' + latex_ω_units[-1])
ax.set_xlim(-950, 950)
ax.set_ylim(-5e-7, 1e-6)
ax.hlines([0], -1e3, 1e3, color=black, alpha=0.1)
ax.vlines([0], -6e-7, 11e-7, color=black, alpha=0.1)
ax.vlines(Δ / kHz, -6e-7, 11e-7, color=red, alpha=0.2, ls='--', label=r'$\Delta$')
ax.plot(full_ω_range / kHz, np.abs(vals_χC_star), label='Abs', lw=2, alpha=0.7, color=green)
ax.plot(full_ω_range / kHz, np.real(vals_χC_star), label='Re', lw=2, alpha=0.7, color=blue)
ax.plot(full_ω_range / kHz, np.imag(vals_χC_star), label='Im', lw=2, alpha=0.7, color=orange)
ax.legend()

ax = ax_dict['3']
ax.set_aspect('equal')
ax.set_xlabel('Real part' + latex_ω_units[-1])
ax.set_ylabel('Imag. part' + latex_ω_units[-1])
xlims, ylims = get_latex_doc_lims(2, 1e-7)
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)
ax.plot(np.real(vals_χC), np.imag(vals_χC), color=pink, lw=3, alpha=0.4, label=r'$\chi_\mathrm{c}(\omega)$')
ax.plot(np.real(vals_χC_star), np.imag(vals_χC_star), color=purple, lw=2, alpha=0.5, ls='--', label=r'$\chi_\mathrm{c}^*(-\omega)$')

resonant_val = mtrap.cavity.χC(-Δ)
ax.scatter(np.real(resonant_val), np.imag(resonant_val), color=blue, alpha=0.4, marker='x', label=r'$\chi_\mathrm{c}(\omega=-\Delta)$')
resonant_val = np.conjugate(mtrap.cavity.χC(-Δ))
ax.scatter(np.real(resonant_val), np.imag(resonant_val), color=red, alpha=0.4, marker='x', label=r'$\chi_\mathrm{c}^*(-\omega=\Delta)$')
ax.legend()

plt.show()


## Difference of susceptibilities $\eta_\mathrm{c} \equiv \chi_\mathrm{c}(\omega) - \chi_\mathrm{c}^*(-\omega)$
- Units: [$\eta_\mathrm{c}$] = ($2\pi \cdot$ Hz)$^{-1}$

Notes:
- Distinctive double loop shape
- Resonant values at ±Re extremes of loop
- Maximum and zero values of Im are off-resonant
- Maximum values of Abs are off-resonant


In [ ]:
mosaic = [['1'], ['2'], ['3']]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

vals = [mtrap.cavity.ηm(ω) for ω in full_ω_range]

ax = ax_dict['1']
ax.set_title(r'$\eta_\mathrm{c} \equiv \chi_\mathrm{c}(\omega) - \chi_\mathrm{c}^*(-\omega)$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\eta_\mathrm{c}$' + latex_ω_units[-1])
ax.set_xlim(-950, 950)
ax.set_ylim(-1e-6, 1e-6)
ax.hlines([0], -1e3, 1e3, color=black, alpha=0.1)
ax.vlines([0], -11e-7, 11e-7, color=black, alpha=0.1)
Δ = mtrap.cavity.Δ
ax.vlines(Δ / kHz, -11e-7, 11e-7, color=red, alpha=0.3, ls='--', label=rf'$\Delta$')
ax.vlines(-Δ / kHz, -11e-7, 11e-7, color=blue, alpha=0.3, ls='--', label=rf'$-\Delta$')
# ax.plot(full_ω_range / kHz, np.abs(vals), label='Abs', lw=2, alpha=0.7, color=green)
ax.scatter(full_ω_range / kHz, np.abs(vals), label='Abs', marker='.', s=10, c=full_ω_range, cmap='gist_rainbow', alpha=0.7)
ax.plot(full_ω_range / kHz, np.real(vals), label='Re', lw=2, alpha=0.7, color=blue)
ax.plot(full_ω_range / kHz, np.imag(vals), label='Im', lw=2, alpha=0.7, color=orange)
ax.legend()

ax = ax_dict['2']
ax.set_aspect('equal')
ax.set_xlabel(r'Re($\eta_\mathrm{c}$)' + latex_ω_units[-1])
ax.set_ylabel(r'Im($\eta_\mathrm{c}$)' + latex_ω_units[-1])
xlims, ylims = get_latex_doc_lims(3, 1e-7)
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)
ax.plot(np.real(vals), np.imag(vals), color=pink, lw=2, alpha=0.7)

resonant_val = mtrap.cavity.ηm(Δ)
ax.scatter(np.real(resonant_val), np.imag(resonant_val), color=red, alpha=0.4, marker='x', label=r'$\eta_\mathrm{c}(\Delta)$')
resonant_val = mtrap.cavity.ηm(-Δ)
ax.scatter(np.real(resonant_val), np.imag(resonant_val), color=blue, alpha=0.4, marker='x', label=r'$\eta_\mathrm{c}(-\Delta)$')
ax.legend()

ax = ax_dict['3']
ax.set_aspect('equal')
ax.set_xlabel(r'Re($\eta_\mathrm{c}$)' + latex_ω_units[-1])
ax.set_ylabel(r'Im($\eta_\mathrm{c}$)' + latex_ω_units[-1])
xlims, ylims = get_latex_doc_lims(3, 1e-7)
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)
ax.scatter(np.real(vals), np.imag(vals), marker='.', s=10, c=full_ω_range, cmap='gist_rainbow', alpha=0.7)

plt.show()


## Product of susceptibilities $\xi_\mathrm{c} \equiv \chi_\mathrm{c}(\omega) \chi_\mathrm{c}^*(-\omega)$
- Units: [$\xi_\mathrm{c}$] = ($2\pi \cdot$ Hz)$^{-2}$

Notes:
- Rotated version of $\eta_\mathrm{c}$
- Resonant values are ±Im peaks

In [ ]:
mosaic = [['1'], ['2']]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

vals = [mtrap.cavity.ξC(ω) for ω in full_ω_range]

ax = ax_dict['1']
ax.set_title(r'$\xi_\mathrm{c} \equiv \chi_\mathrm{c}(\omega) \chi_\mathrm{c}^*(-\omega)$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\xi_\mathrm{c}$' + latex_ω_units[-2])
ax.set_xlim(-950, 950)
ax.set_ylim(-2e-13, 2e-13)
ax.hlines([0], -1e3, 1e3, color=black, alpha=0.1)
ax.vlines([0], -2e-13, 2e-13, color=black, alpha=0.1)
Δ = mtrap.cavity.Δ
ax.vlines(Δ / kHz, -3e-13, 3e-13, color=red, alpha=0.3, ls='--', label=rf'$\Delta$')
ax.vlines(-Δ / kHz, -3e-13, 3e-13, color=blue, alpha=0.3, ls='--', label=rf'$-\Delta$')
ax.plot(full_ω_range / kHz, np.abs(vals), label='Abs', lw=2, alpha=0.7, color=green)
ax.plot(full_ω_range / kHz, np.real(vals), label='Re', lw=2, alpha=0.7, ls='-', color=blue)
ax.plot(full_ω_range / kHz, np.imag(vals), label='Im', lw=2, alpha=0.7, ls='-', color=orange)
ax.legend(loc='lower right')

ax = ax_dict['2']
ax.set_aspect('equal')
ax.set_xlabel(r'Re($\xi_\mathrm{c}$)' + latex_ω_units[-2])
ax.set_ylabel(r'Im($\xi_\mathrm{c}$)' + latex_ω_units[-2])
xlims, ylims = get_latex_doc_lims(7, 1e-14)
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)
ax.plot(np.real(vals), np.imag(vals), color=pink, lw=2, alpha=0.7)

resonant_val = mtrap.cavity.ξC(Δ)
ax.scatter(np.real(resonant_val), np.imag(resonant_val), color=red, alpha=0.4, marker='x', label=r'$\xi_\mathrm{c}(\Delta)$')
resonant_val = mtrap.cavity.ξC(-Δ)
ax.scatter(np.real(resonant_val), np.imag(resonant_val), color=blue, alpha=0.4, marker='x', label=r'$\xi_\mathrm{c}(-\Delta)$')
ax.legend()

plt.show()


## Verify difference $\propto$ product: $\eta_\mathrm{c} = 2i \Delta \xi_\mathrm{c}$

Notes:
- Reduces units to just inverse angular frequency
- Rotates by 90° as multiplied by $i$, as expected

In [ ]:
mosaic = [['1'], ['2']]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

Δ = mtrap.cavity.Δ
vals = 2j * Δ * np.array([mtrap.cavity.ξC(ω) for ω in full_ω_range])

ax = ax_dict['1']
ax.set_title(r'$\eta_\mathrm{c} = 2i \Delta \xi_\mathrm{c}$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$2i \Delta \xi_\mathrm{c}$' + latex_ω_units[-1])
ax.set_xlim(-950, 950)
ax.set_ylim(-1e-6, 1e-6)
ax.hlines([0], -1e3, 1e3, color=black, alpha=0.1)
ax.vlines([0], -11e-7, 11e-7, color=black, alpha=0.1)
ax.vlines(Δ / kHz, -11e-7, 11e-7, color=red, alpha=0.3, ls='--', label=rf'$\Delta$')
ax.vlines(-Δ / kHz, -11e-7, 11e-7, color=blue, alpha=0.3, ls='--', label=rf'$-\Delta$')
ax.plot(full_ω_range / kHz, np.abs(vals), label='Abs', lw=2, alpha=0.7, color=green)
ax.plot(full_ω_range / kHz, np.real(vals), label='Re', lw=2, alpha=0.7, color=blue)
ax.plot(full_ω_range / kHz, np.imag(vals), label='Im', lw=2, alpha=0.7, color=orange)
ax.legend()

ax = ax_dict['2']
ax.set_aspect('equal')
ax.set_xlabel(r'Re($2i \Delta \xi_\mathrm{c}$)' + latex_ω_units[-1])
ax.set_ylabel(r'Im($2i \Delta \xi_\mathrm{c}$)' + latex_ω_units[-1])
xlims, ylims = get_latex_doc_lims(3, 1e-7)
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)
ax.plot(np.real(vals), np.imag(vals), color=pink, lw=2, alpha=0.7)

resonant_val = 2j * Δ * mtrap.cavity.ξC(Δ)
ax.scatter(np.real(resonant_val), np.imag(resonant_val), color=red, alpha=0.4, marker='x', label=r'$2i \Delta \xi_\mathrm{c}(\Delta)$')
resonant_val = 2j * Δ * mtrap.cavity.ξC(-Δ)
ax.scatter(np.real(resonant_val), np.imag(resonant_val), color=blue, alpha=0.4, marker='x', label=r'$2i \Delta \xi_\mathrm{c}(-\Delta)$')
ax.legend()

plt.show()


# PART 1b: Mechanical susceptibility functions
- Units: [$\chi_\mathrm{c}(\omega)$] = ($2\pi\cdot$ Hz)$^{-1}$

Notes:
- Very very thin resonant peaks, since the mechanical damping is proprotional to the very low pressure $\Gamma_j \propto P_\mathrm{gas}$ e.g. $\Gamma_j \sim 10^{-6}$ kHz at $10^{-6}$ mbar pressure

In [ ]:
sl = slice(0, 3)

dims_to_loop_over = list(mtrap.dimensions.values())[sl]

mosaic = [[dim_j.label] for dim_j in dims_to_loop_over]
mosaic = mosaic + [[dim_j.label + 'star'] for dim_j in dims_to_loop_over]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.7})

for dim_j in dims_to_loop_over:
    j, n = dim_j.label[0], dim_j.label[1:]
    jn_string = rf'{j}_{n}'
    Ωj = dim_j.Ω

    # Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
    width = 1e-5 * kHz
    step = 1e-8 * kHz

    ω_range_around_resonance = np.arange(Ωj - width, Ωj + width + step, step)
    vals_χj = [dim_j.χ(ω) for ω in ω_range_around_resonance]

    ω_range_around_negative_resonance = np.arange(-Ωj - width, -Ωj + width + step, step)
    vals_χj_star = [np.conjugate(dim_j.χ(-ω)) for ω in ω_range_around_negative_resonance]

    ax = ax_dict[dim_j.label]
    ax.set_title(rf'$\chi_{{{jn_string}}}(\omega) \equiv [i(\omega - \Omega_{{{jn_string}}}) + \frac{{\Gamma_{{{jn_string}}}}}{{2}}]^{{-1}}$')
    ax.set_xlabel('Frequency \n around ' + rf'${{{jn_string}}}$ resonance' + '\n [kHz]')
    ax.set_ylabel(rf'$\chi_{{{jn_string}}}(\omega)$' + latex_ω_units[-1])
    xlims = ω_range_around_resonance.min() / kHz, ω_range_around_resonance.max() / kHz
    ax.set_xlim(*xlims)
    ax.set_ylim(-350, 350)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines(Ωj / kHz, -350, 350, color=black, alpha=0.2, ls='--', label=rf'$\Omega_{{{jn_string}}}$')
    ax.plot(ω_range_around_resonance / kHz, np.abs(vals_χj), label='Abs', lw=4, alpha=0.4, color=green)
    ax.plot(ω_range_around_resonance / kHz, np.real(vals_χj), label='Re', lw=2, alpha=0.7,color=blue)
    ax.plot(ω_range_around_resonance / kHz, np.imag(vals_χj), label='Im', lw=1, alpha=0.7, color=orange)
    ax.legend(loc='lower right')

    ax = ax_dict[dim_j.label + 'star']
    ax.set_title(rf'$\chi^*_{{{jn_string}}}(-\omega) \equiv [i(\omega + \Omega_{{{jn_string}}}) + \frac{{\Gamma_{{{jn_string}}}}}{{2}}]^{{-1}}$')
    ax.set_xlabel('Frequency \n around ' + rf'${{{jn_string}}}$ resonance' + '\n [kHz]')
    ax.set_ylabel(rf'$\chi^*_{{{jn_string}}}(-\omega)$' + latex_ω_units[-1])
    xlims = ω_range_around_negative_resonance.min() / kHz, ω_range_around_negative_resonance.max() / kHz
    ax.set_xlim(*xlims)
    ax.set_ylim(-350, 350)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines(-Ωj / kHz, -350, 350, color=black, alpha=0.2, ls='--', label=rf'$-\Omega_{{{jn_string}}}$')
    ax.plot(ω_range_around_negative_resonance / kHz, np.abs(vals_χj_star), label='Abs', lw=4, alpha=0.4, color=green)
    ax.plot(ω_range_around_negative_resonance / kHz, np.real(vals_χj_star), label='Re', lw=2, alpha=0.7,color=blue)
    ax.plot(ω_range_around_negative_resonance / kHz, np.imag(vals_χj_star), label='Im', lw=1, alpha=0.7, color=orange)
    ax.legend(loc='lower right')

plt.show()


# PART 2a: Non-phase-dependent collective mechanical susceptibility function $\mathcal{D}_0$

## $\mathcal{D}_0 \equiv \sum_j g_j^2 \mu_j $
- Units: [$\mathcal{D}_0$] = $2\pi\cdot$ Hz

Notes:
- Peaks at resonant frequencies $\Omega_j$
- Peaks scaled by frequency, due my choice of $g_j$ being larger for larger frequencies

In [ ]:
mosaic = [['1'], ['2'], ['3'], ['4']]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

full_vals = [mtrap.D(0, ω) for ω in full_ω_range]
vals = [mtrap.D(0, ω) for ω in ω_range]

ax = ax_dict['1']
ax.set_title(r'$\mathcal{D}_0 \equiv \sum_j g_j^2 \mu_j$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\mathcal{D}_0$' + latex_ω_units[1])
ax.set_xlim(-950, 950)
ax.plot(full_ω_range / kHz, np.abs(full_vals), label='Abs', lw=3, alpha=0.4, color=green)
ax.plot(full_ω_range / kHz, np.real(full_vals), label='Re', lw=2, alpha=0.4, ls='--', color=blue)
ax.plot(full_ω_range / kHz, np.imag(full_vals), label='Im', lw=1, alpha=0.4, ls='--', color=orange)
ax.legend()

ax = ax_dict['2']
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\mathcal{D}_0$' + latex_ω_units[1])
ax.set_xlim(-950, 950)
ax.set_ylim(1e-3, 1e14)
ax.semilogy(full_ω_range / kHz, np.abs(full_vals), label='Abs', lw=3, alpha=0.4, color=green)
ax.semilogy(full_ω_range / kHz, np.abs(np.real(full_vals)), label='|Re|', lw=2, alpha=0.4, ls='--', color=blue)
ax.semilogy(full_ω_range / kHz, np.abs(np.imag(full_vals)), label='|Im|', lw=1, alpha=0.4, ls='--', color=orange)
ax.legend()

ax = ax_dict['3']
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\mathcal{D}_0$' + latex_ω_units[1])
ax.set_xlim(280, 320)
ax.set_ylim(-2.5e13, 2.5e13)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], -3e13, 3e13, color=black, alpha=0.2, ls='--', label=r'All $\Omega_j$')
ax.plot(ω_range / kHz, np.abs(vals), label='Abs', lw=3, alpha=0.4, color=green)
ax.plot(ω_range / kHz, np.real(vals), label='Re', lw=2, alpha=0.4, ls='--', color=blue)
ax.plot(ω_range / kHz, np.imag(vals), label='Im', lw=1, alpha=0.4, ls='--', color=orange)
ax.legend()

ax = ax_dict['4']
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\mathcal{D}_0$' + latex_ω_units[1])
ax.set_xlim(280, 320)
ax.set_ylim(1e-3, 1e14)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e-5, 1e15, color=black, alpha=0.2, ls='--', label=r'All $\Omega_j$')
ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=3, alpha=0.4, color=green)
ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.4, ls='--', color=blue)
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.4, ls='--', color=orange)
ax.legend()

plt.show()


## Dimensionless $\eta_\mathrm{c} \mathcal{D}_0 \equiv \eta_\mathrm{c} \sum_j g_j^2 \mu_j$
- Units: [$\eta_\mathrm{c} \mathcal{D}_0$] = 1

Notes:
- Peaks at resonant frequencies $\Omega_j$
- Peaks scaled by frequency, slightly more pronounced by $\eta_\mathrm{c}$ multiplication

In [ ]:
mosaic = [['1'], ['2'], ['3'], ['4']]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

full_vals = [mtrap.cavity.ηm(ω) * mtrap.D(0, ω) for ω in full_ω_range]
vals = [mtrap.cavity.ηm(ω) * mtrap.D(0, ω) for ω in ω_range]

ax = ax_dict['1']
ax.set_title(r'$\eta_\mathrm{c} \mathcal{D}_0 \equiv \eta_\mathrm{c} \sum_j g_j^2 \mu_j$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\eta_\mathrm{c} \mathcal{D}_0$')
ax.set_xlim(-950, 950)
ax.set_ylim(-2e7, 2e7)
ax.plot(full_ω_range / kHz, np.abs(full_vals), label='Abs', lw=3, alpha=0.4, color=green)
ax.plot(full_ω_range / kHz, np.real(full_vals), label='Re', lw=2, alpha=0.4, ls='--', color=blue)
ax.plot(full_ω_range / kHz, np.imag(full_vals), label='Im', lw=1, alpha=0.4, ls='--', color=orange)
ax.legend()

ax = ax_dict['2']
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\eta_\mathrm{c} \mathcal{D}_0$')
ax.set_xlim(-950, 950)
ax.set_ylim(1e-8, 1e8)
ax.semilogy(full_ω_range / kHz, np.abs(full_vals), label='Abs', lw=3, alpha=0.4, color=green)
ax.semilogy(full_ω_range / kHz, np.abs(np.real(full_vals)), label='|Re|', lw=2, alpha=0.4, ls='--', color=blue)
ax.semilogy(full_ω_range / kHz, np.abs(np.imag(full_vals)), label='|Im|', lw=1, alpha=0.4, ls='--', color=orange)
ax.legend()

ax = ax_dict['3']
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\eta_\mathrm{c} \mathcal{D}_0$')
ax.set_xlim(280, 320)
ax.set_ylim(-2e7, 2e7)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], -3e7, 3e7, color=black, alpha=0.2, ls='--', label=r'All $\Omega_j$')
ax.plot(ω_range / kHz, np.abs(vals), label='Abs', lw=3, alpha=0.4, color=green)
ax.plot(ω_range / kHz, np.real(vals), label='Re', lw=2, alpha=0.4, ls='--', color=blue)
ax.plot(ω_range / kHz, np.imag(vals), label='Im', lw=1, alpha=0.4, ls='--', color=orange)
ax.legend()

ax = ax_dict['4']
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\eta_\mathrm{c} \mathcal{D}_0$')
ax.set_xlim(280, 320)
ax.set_ylim(1e-8, 1e8)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e-9, 1e9, color=black, alpha=0.2, ls='--', label=r'All $\Omega_j$')
ax.semilogy(ω_range / kHz, [np.abs(mtrap.cavity.ηm(ω)* mtrap.D(0, ω)) for ω in ω_range], label='Abs', lw=3, alpha=0.4, color=green)
ax.semilogy(ω_range / kHz, [np.abs(np.real(mtrap.cavity.ηm(ω) * mtrap.D(0, ω))) for ω in ω_range], label='|Re|', lw=2, alpha=0.4, ls='--', color=blue)
ax.semilogy(ω_range / kHz, [np.abs(np.imag(mtrap.cavity.ηm(ω)* mtrap.D(0, ω))) for ω in ω_range], label='|Im|', lw=1, alpha=0.4, ls='--', color=orange)
ax.legend()

plt.show()


## Complex parts of $\mathcal{D}_0$ and $\eta_\mathrm{c} \mathcal{D}_0$ around $j^\mathrm{th}$ resonance
Notes:
- multiplication by $\eta_\mathrm{c}$ rotates the complex resonant peaks downwards

In [ ]:
mosaic = [
    ['D0'],
    ['eta D0'],
]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.4})
fig.suptitle(r'Complex values around $j^\mathrm{th}$ resonance')

ax = ax_dict['D0']
ax.set_xlabel(r'$\mathrm{Re}(\mathcal{D}_0)$' + latex_ω_units[1])
ax.set_ylabel(r'$\mathrm{Im}(\mathcal{D}_0)$' + latex_ω_units[1])
ax.set_aspect('equal')
xlims, ylims = get_latex_doc_lims(5, 1e12)
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.2)
ax.vlines([0], *ylims, color=black, alpha=0.2)
ax.set_title(r'$\mathcal{D}_0$')

ax = ax_dict['eta D0']
ax.set_xlabel(r'$\mathrm{Re}(\eta_\mathrm{c} \mathcal{D}_0)$')
ax.set_ylabel(r'$\mathrm{Im}(\eta_\mathrm{c} \mathcal{D}_0)$')
ax.set_aspect('equal')
xlims, ylims = get_latex_doc_lims(6, 1e6)
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.2)
ax.vlines([0], *ylims, color=black, alpha=0.2)
ax.set_title(r'$\eta_\mathrm{c} \mathcal{D}_0$')

# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 1e-8 * kHz
for dim in mtrap.dimensions.values():
    ω_range_around_resonance = np.arange(dim.Ω - width, dim.Ω + width + step, step)
    jn = dim.label
    jn_string = f'{jn[0]}_{jn[1:]}'

    ax = ax_dict['D0']
    vals = [mtrap.D(0, ω) for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$', loc='center left')

    ax = ax_dict['eta D0']
    vals = [mtrap.cavity.ηm(ω) * mtrap.D(0, ω) for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$', loc='center left')

plt.show()


## Complex parts of $\mathcal{D}_0^2$ and $\mathcal{D}_2 \mathcal{D}_{-2}$, and their product with $\xi_\mathrm{c}$

Notes:
- Distinctive cardiod shape when the complex circle is squared
- Multiplication by $\xi_\mathrm{c}$ rotates the cardiod upwards
- Values are nearly identical, only differ by 1 part in 1-10 million!

In [ ]:
mosaic = [
    ['D0^2'],
    ['D2_D-2'],
    ['1 - D2_D-2 / D0^2'],
    ['xi D0^2'],
    ['xi D2_D-2'],
]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.4})

xlims, ylims = get_latex_doc_lims(1.5, 1e26)
ax = ax_dict['D0^2']
ax.set_title(
    r'Complex values around $j^\mathrm{th}$ resonance'
    '\n'
    r'$\mathcal{D}_0^2$'
)
ax.set_xlabel(r'$\mathrm{Re}(\mathcal{D}_0^2)$' + latex_ω_units[2])
ax.set_ylabel(r'$\mathrm{Im}(\mathcal{D}_0^2)$' + latex_ω_units[2])
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['D2_D-2']
ax.set_title(r'$\mathcal{D}_2 \mathcal{D}_{-2}$')
ax.set_xlabel(r'$\mathrm{Re}(\mathcal{D}_2 \mathcal{D}_{-2})$' + latex_ω_units[2])
ax.set_ylabel(r'$\mathrm{Im}(\mathcal{D}_2 \mathcal{D}_{-2})$' + latex_ω_units[2])
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['1 - D2_D-2 / D0^2']
ax.set_title(r'$1 - \frac{\mathcal{D}_2 \mathcal{D}_{-2}}{\mathcal{D}_0^2}$')
ax.set_xlabel(r'$\mathrm{Re} \left( 1 - \frac{\mathcal{D}_2 \mathcal{D}_{-2}}{\mathcal{D}_0^2} \right)$')
ax.set_ylabel(r'$\mathrm{Im} \left( 1 - \frac{\mathcal{D}_2 \mathcal{D}_{-2}}{\mathcal{D}_0^2} \right)$')
xlims = -2e-5, 2e-5
ylims = -1e-6, 1e-6
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

xlims, ylims = get_latex_doc_lims(2.1, 1e13)
ax = ax_dict['xi D0^2']
ax.set_title(r'$\xi_\mathrm{c} \mathcal{D}_0^2$')
ax.set_xlabel(r'$\mathrm{Re}(\xi_\mathrm{c} \mathcal{D}_0^2)$')
ax.set_ylabel(r'$\mathrm{Im}(\xi_\mathrm{c} \mathcal{D}_0^2)$')
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['xi D2_D-2']
ax.set_title(r'$\xi_\mathrm{c} \mathcal{D}_2 \mathcal{D}_{-2}$')
ax.set_xlabel(r'$\mathrm{Re}(\xi_\mathrm{c} \mathcal{D}_2 \mathcal{D}_{-2})$')
ax.set_ylabel(r'$\mathrm{Im}(\xi_\mathrm{c} \mathcal{D}_2 \mathcal{D}_{-2})$')
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 1e-8 * kHz
for dim in mtrap.dimensions.values():
    ω_range_around_resonance = np.arange(dim.Ω - width, dim.Ω + width + step, step)
    jn = dim.label
    jn_string = f'{jn[0]}_{jn[1:]}'

    ax = ax_dict['D0^2']
    vals = [mtrap.D(0, ω)**2 for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$', loc='center left')

    ax = ax_dict['D2_D-2']
    vals = [mtrap.D(2, ω) * mtrap.D(-2, ω) for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$', loc='center left')

    ax = ax_dict['1 - D2_D-2 / D0^2']
    vals = [1 - mtrap.D(2, ω) * mtrap.D(-2, ω) / mtrap.D(0, ω)**2 for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$', loc='center left')

    ax = ax_dict['xi D0^2']
    vals = [mtrap.cavity.ξC(ω) * mtrap.D(0, ω)**2 for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$', loc='center left')

    ax = ax_dict['xi D2_D-2']
    vals = [mtrap.cavity.ξC(ω) * mtrap.D(2, ω) * mtrap.D(-2, ω) for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$', loc='center left')

plt.show()


# PART 2b: Phase-dependent collective mechanical susceptiblity $\mathcal{P}$

## Definition using $\mathcal{D}_n$'s: $\mathcal{P} \equiv \mathcal{D}_0^2 - \mathcal{D}_{2}\mathcal{D}_{-2}$
- Units: [$\mathcal{P}$] = ($2\pi\cdot$ Hz)$^2$

Notes:
- On resonant peaks still
- Middle frequency peaks highly surpressed, due to average phase difference being smallest
- Edge (highest, lowest) frequency peaks pronounced, due to average phase difference being highest
- Asymmetry of imaginary part around middle of frequency spread
- Imaginary part is much greater than the real part

In [ ]:
mosaic = [['1'], ['2']]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

vals = [mtrap.P(ω) for ω in ω_range]

ax = ax_dict['1']
ax.set_title(r'$\mathcal{P} \equiv \mathcal{D}_0^2 - \mathcal{D}_{2}\mathcal{D}_{-2}$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\mathcal{P}$' + latex_ω_units[2])
ax.set_xlim(280, 320)
ax.set_ylim(-2.5e20, 2.5e20)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], -3e20, 3e20, label=r'All $\Omega_j$', color=black, alpha=0.2, ls='--')
ax.plot(ω_range / kHz, np.real(vals), label='Re', lw=2, alpha=0.7, ls='--')
ax.plot(ω_range / kHz, np.imag(vals), label='Im', lw=1, alpha=0.7, ls='--')
ax.plot(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
ax.legend()

ax = ax_dict['2']
ax.set_title(r'$\mathcal{P} \equiv \mathcal{D}_0^2 - \mathcal{D}_{2}\mathcal{D}_{-2}$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\mathcal{P}$' + latex_ω_units[2])
ax.set_xlim(280, 320)
ax.set_ylim(1e1, 1e21)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e0, 1e22, label=r'All $\Omega_j$', color=black, alpha=0.2, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
ax.legend()

plt.show()


In [ ]:
mosaic = [['1'], ['2']]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

vals = [mtrap.D(0, ω)**2 for ω in ω_range]

ax = ax_dict['1']
ax.set_title(r'$\mathcal{D}_0^2$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\mathcal{D}_0^2$' + latex_ω_units[2])
ax.set_xlim(280, 320)
ax.set_ylim(-1e26, 5e26)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], -6e26, 6e26, label=r'All $\Omega_j$', color=black, alpha=0.2, ls='--')
ax.plot(ω_range / kHz, np.real(vals), label='Re', lw=2, alpha=0.7, ls='--')
ax.plot(ω_range / kHz, np.imag(vals), label='Im', lw=1, alpha=0.7, ls='--')
ax.plot(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
ax.legend()

ax = ax_dict['2']
ax.set_title(r'$\mathcal{D}_0^2$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\mathcal{D}_0^2$' + latex_ω_units[2])
ax.set_xlim(280, 320)
ax.set_ylim(1e3, 1e27)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e2, 1e28, label=r'All $\Omega_j$', color=black, alpha=0.2, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
ax.legend()

plt.show()


In [ ]:
mosaic = [['1'], ['2']]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

vals = [mtrap.D(2, ω) * mtrap.D(-2, ω) for ω in ω_range]

ax = ax_dict['1']
ax.set_title(r'$\mathcal{D}_2 \mathcal{D}_{-2}$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\mathcal{D}_2 \mathcal{D}_{-2}$' + latex_ω_units[2])
ax.set_xlim(280, 320)
ax.set_ylim(-1e26, 5e26)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], -6e26, 6e26, label=r'All $\Omega_j$', color=black, alpha=0.2, ls='--')
ax.plot(ω_range / kHz, np.real(vals), label='Re', lw=2, alpha=0.7, ls='--')
ax.plot(ω_range / kHz, np.imag(vals), label='Im', lw=1, alpha=0.7, ls='--')
ax.plot(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
ax.legend()

ax = ax_dict['2']
ax.set_title(r'$\mathcal{D}_2 \mathcal{D}_{-2}$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$\mathcal{D}_2 \mathcal{D}_{-2}$' + latex_ω_units[2])
ax.set_xlim(280, 320)
ax.set_ylim(1e3, 1e27)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e2, 1e28, label=r'All $\Omega_j$', color=black, alpha=0.2, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
ax.legend()

plt.show()


In [ ]:
mosaic = [['1'], ['2']]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

vals = [1 - mtrap.D(2, ω) * mtrap.D(-2, ω) / mtrap.D(0, ω)**2 for ω in ω_range]

ax = ax_dict['1']
ax.set_title(r'$1 - \frac{\mathcal{D}_2 \mathcal{D}_{-2}}{\mathcal{D}_0^2}$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$1 - \frac{\mathcal{D}_2 \mathcal{D}_{-2}}{\mathcal{D}_0^2}$')
ax.set_xlim(280, 320)
ax.set_ylim(-5e9, 5e9)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], -2e9, 6e9, label=r'All $\Omega_j$', color=black, alpha=0.2, ls='--')
ax.plot(ω_range / kHz, np.real(vals), label='Re', lw=2, alpha=0.7, ls='--')
ax.plot(ω_range / kHz, np.imag(vals), label='Im', lw=1, alpha=0.7, ls='--')
ax.plot(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
ax.legend()

ax = ax_dict['2']
ax.set_title(r'$1 - \frac{\mathcal{D}_2 \mathcal{D}_{-2}}{\mathcal{D}_0^2}$')
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(r'$1 - \frac{\mathcal{D}_2 \mathcal{D}_{-2}}{\mathcal{D}_0^2}$')
ax.set_xlim(280, 320)
ax.set_ylim(1e-13, 1e13)
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e-13, 1e13, label=r'All $\Omega_j$', color=black, alpha=0.2, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
ax.legend()

plt.show()


## Verify equivalence of different breakdowns of $\mathcal{P} \equiv \sum_j \mathcal{P}_j \equiv \sum_{jk} \mathcal{P}_{jk}$

---

Definition with $\mathcal{D}_n \equiv \sum_j g_j^2 \mu_j e^{i n \phi_j}$ 

$$
    \mathcal{P} \equiv \mathcal{D}_0^2 - \mathcal{D}_{2}\mathcal{D}_{-2}
$$

---

Sum of contribution from each dimension

$$ \mathcal{P} \equiv \sum_j \mathcal{P}_j $$

$j = 1,\ldots,N$

- Definition 1: 
$$
    \mathcal{P}_j^{(1)}
    \equiv
    2 g_j^2 \mu_j
    \sum_{k \neq j}
        g_k^2 \mu_k \sin^2 (\phi_j - \phi_k)
    \equiv
    g_j^2 \mu_j \mathcal{K}_j^{(1)}
$$

- Definition 2: 
$$
    \mathcal{P}_j^{(2)}
    \equiv
    2 g_j^2 \mu_j
    \sum_k
        g_k^2 \mu_k
        \left( 1 - e^{2i (\phi_j - \phi_k)} \right)
    \equiv
    g_j^2 \mu_j \mathcal{K}_j^{(2)}
$$

---

Sum of contribution from each pair of dimensions $\mathcal{P} \equiv \sum_j \mathcal{P}_{jk}$

$jk \in$ $N \choose 2$ $= N(N-1)/2$ unordered pairs of dimensions $j,k$

$$
    \mathcal{P}_{jk}
    \equiv
    4 g_j^2 \mu_j g_k^2 \mu_k
    \sin^2 (\phi_j - \phi_k)
$$

---

In [ ]:
mosaic = [
    ['1', '2'],
    ['3', '3'],
]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.3})

labels = {
    'P': r'$\mathcal{P} \equiv \mathcal{D}_0^2 - \mathcal{D}_{2}\mathcal{D}_{-2}$',
    'Pj_1': r'$ \mathcal{P} \equiv \sum_j \mathcal{P}_j^{(1)}$',
    'Pj_2': r'$ \mathcal{P} \equiv \sum_j \mathcal{P}_j^{(2)}$',
    'Pjk': r'$\mathcal{P} \equiv \sum_{jk} \mathcal{P}_{jk}$',
}

vals_P = [mtrap.P(ω) for ω in ω_range]
vals_P_via_Pj = [mtrap.P_via_Pj(ω) for ω in ω_range]
vals_P_via_Pj_2 = [mtrap.P_via_Pj_2(ω) for ω in ω_range]
vals_P_via_Pjk = [mtrap.P_via_Pjk(ω) for ω in ω_range]

ax = ax_dict['1']
ax.set_xlim(280, 320)
ax.set_ylim(1e1, 1e21)
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(latex_ω_units[2])
ax.set_title(r'$|\mathrm{Re}(\mathcal{P})|$')
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e0, 1e22, color=black, alpha=0.2, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.real(vals_P)), label=labels['P'], lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(np.real(vals_P_via_Pj)), label=labels['Pj_1'], lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.real(vals_P_via_Pj_2)), label=labels['Pj_2'], lw=1, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.real(vals_P_via_Pjk)), label=labels['Pjk'], lw=0.5, alpha=0.6, ls='--')

ax = ax_dict['2']
ax.set_xlim(280, 320)
ax.set_ylim(1e1, 1e21)
ax.set_xlabel('Frequency [kHz]')
ax.set_title(r'$|\mathrm{Im}(\mathcal{P})|$')
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e0, 1e22, color=black, alpha=0.2, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals_P)), label=labels['P'], lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals_P_via_Pj)), label=labels['Pj_1'], lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals_P_via_Pj_2)), label=labels['Pj_2'], lw=1, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals_P_via_Pjk)), label=labels['Pjk'], lw=0.5, alpha=0.6, ls='--')

ax = ax_dict['3']
ax.set_xlim(280, 320)
ax.set_ylim(1e9, 1e21)
ax.set_xlabel('Frequency [kHz]')
ax.set_ylabel(latex_ω_units[2])
ax.set_title(r'$|\mathcal{P}|$')
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e0, 1e22, color=black, alpha=0.2, ls='--')
ax.semilogy(ω_range / kHz, np.abs(vals_P), label=labels['P'], lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(vals_P_via_Pj), label=labels['Pj_1'], lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(vals_P_via_Pj_2), label=labels['Pj_2'], lw=1, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(vals_P_via_Pjk), label=labels['Pjk'], lw=0.5, alpha=0.6, ls='--')
ax.legend(loc='lower right')

plt.show()


## Dimensionless $\xi_\mathrm{c} \mathcal{P}$
- Units: [$\xi_\mathrm{c} \mathcal{P}$] = 1

In [ ]:
mosaic = [
    ['1', '2'],
    ['3', '3'],
]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

labels = {
    'P': r'$\xi_\mathrm{c} \mathcal{P} \equiv \xi_\mathrm{c}(\mathcal{D}_0^2 - \mathcal{D}_{2}\mathcal{D}_{-2})$',
    'Pj_1': r'$\xi_\mathrm{c} \mathcal{P} \equiv \xi_\mathrm{c} \sum_j \mathcal{P}_j^{(1)}$',
    'Pj_2': r'$\xi_\mathrm{c} \mathcal{P} \equiv \xi_\mathrm{c} \sum_j \mathcal{P}_j^{(2)}$',
    'Pjk': r'$\xi_\mathrm{c} \mathcal{P} \equiv \xi_\mathrm{c} \sum_{jk} \mathcal{P}_{jk}$',
}

vals_P = [mtrap.cavity.ξC(ω) * mtrap.P(ω) for ω in ω_range]
vals_P_via_Pj = [mtrap.cavity.ξC(ω) * mtrap.P_via_Pj(ω) for ω in ω_range]
vals_P_via_Pj_2 = [mtrap.cavity.ξC(ω) * mtrap.P_via_Pj_2(ω) for ω in ω_range]
vals_P_via_Pjk = [mtrap.cavity.ξC(ω) * mtrap.P_via_Pjk(ω) for ω in ω_range]

ax = ax_dict['1']
ax.set_xlim(280, 320)
ax.set_ylim(1e-3, 1e8)
ax.set_xlabel('Frequency [kHz]')
ax.set_title(r'$|\mathrm{Re}(\xi_\mathrm{c} \mathcal{P})|$')
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e-4, 1e9, color=black, alpha=0.2, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.real(vals_P)), label=labels['P'], lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(np.real(vals_P_via_Pj)), label=labels['Pj_1'], lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.real(vals_P_via_Pj_2)), label=labels['Pj_2'], lw=1, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.real(vals_P_via_Pjk)), label=labels['Pjk'], lw=0.5, alpha=0.6, ls='--')

ax = ax_dict['2']
ax.set_xlim(280, 320)
ax.set_ylim(1e-3, 1e8)
ax.set_xlabel('Frequency [kHz]')
ax.set_title(r'$|\mathrm{Im}(\xi_\mathrm{c} \mathcal{P})|$')
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e-4, 1e9, color=black, alpha=0.2, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals_P)), label=labels['P'], lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals_P_via_Pj)), label=labels['Pj_1'], lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals_P_via_Pj_2)), label=labels['Pj_2'], lw=1, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(np.imag(vals_P_via_Pjk)), label=labels['Pjk'], lw=0.5, alpha=0.6, ls='--')

ax = ax_dict['3']
ax.set_xlim(280, 320)
ax.set_ylim(1e-3, 1e8)
ax.set_xlabel('Frequency [kHz]')
ax.set_title(r'$|\xi_\mathrm{c} \mathcal{P}|$')
ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e-4, 1e9, color=black, alpha=0.2, ls='--')
ax.semilogy(ω_range / kHz, np.abs(vals_P), label=labels['P'], lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(vals_P_via_Pj), label=labels['Pj_1'], lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(vals_P_via_Pj_2), label=labels['Pj_2'], lw=1, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(vals_P_via_Pjk), label=labels['Pjk'], lw=0.5, alpha=0.6, ls='--')
ax.legend(loc='lower right')

plt.show()


## Complex parts of $\mathcal{P}$ around $j^\mathrm{th}$ resonance

Notes:
- $\mathcal{P}$ itself is vertically aligned
- Multiplication by $\xi_\mathrm{c}$ rotates it

In [ ]:
mosaic = [
    ['P'],
    ['P_via_Pj'],
    ['P_via_Pj_2'],
    ['P_via_Pjk'],
    ['xi P'],
    ['D0'],
    ['eta D0'],
]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

xlims, ylims = get_latex_doc_lims(8, 1e19)

ax = ax_dict['P']
ax.set_title(
    r'Complex values around $j^\mathrm{th}$ resonance'
    '\n'
    r'$\mathcal{P}$'
)
ax.set_xlabel(r'$\mathrm{Re}(\mathcal{P})$' + latex_ω_units[2])
ax.set_ylabel(r'$\mathrm{Im}(\mathcal{P})$' + latex_ω_units[2])
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['P_via_Pj']
ax.set_title(r'$\mathcal{P} \equiv 2 \sum_j g_j^2 \mu_j \sum_{k \neq j} g_k^2 \mu_k \sin^2(\phi_j - \phi_k)$')
ax.set_xlabel(r'$\mathrm{Re}(\mathcal{P})$' + latex_ω_units[2])
ax.set_ylabel(r'$\mathrm{Im}(\mathcal{P})$' + latex_ω_units[2])
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['P_via_Pj_2']
ax.set_title(r'$\mathcal{P} \equiv \sum_j g_j^2 \mu_j \sum_{k} g_k^2 \mu_k \left(e^{2i(\phi_j - \phi_k)} - 1 \right)$')
ax.set_xlabel(r'$\mathrm{Re}(\mathcal{P})$' + latex_ω_units[2])
ax.set_ylabel(r'$\mathrm{Im}(\mathcal{P})$' + latex_ω_units[2])
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['P_via_Pjk']
ax.set_title(r'$\mathcal{P} \equiv 4 \sum_{jk} g_j^2 \mu_j g_k^2 \mu_k \sin^2(\phi_j - \phi_k)$')
ax.set_xlabel(r'$\mathrm{Re}(\mathcal{P})$' + latex_ω_units[2])
ax.set_ylabel(r'$\mathrm{Im}(\mathcal{P})$' + latex_ω_units[2])
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['xi P']
ax.set_title(r'$\xi_\mathrm{c} \mathcal{P}$')
ax.set_xlabel(r'$\mathrm{Re}(\xi_\mathrm{c} \mathcal{P})$')
ax.set_ylabel(r'$\mathrm{Im}(\xi_\mathrm{c} \mathcal{P})$')
ax.set_aspect('equal')
xlims, ylims = get_latex_doc_lims(1, 1e7)
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['D0']
ax.set_title(r'$\mathcal{D}_0$')
ax.set_xlabel(r'$\mathrm{Re}(\mathcal{D}_0)$' + latex_ω_units[1])
ax.set_ylabel(r'$\mathrm{Im}(\mathcal{D}_0)$' + latex_ω_units[1])
ax.set_aspect('equal')
xlims, ylims = get_latex_doc_lims(6, 1e12)
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['eta D0']
ax.set_title(r'$\eta_\mathrm{c} \mathcal{D}_0$')
ax.set_xlabel(r'$\mathrm{Re}(\eta_\mathrm{c} \mathcal{D}_0)$')
ax.set_ylabel(r'$\mathrm{Im}(\eta_\mathrm{c} \mathcal{D}_0)$')
xlims, ylims = get_latex_doc_lims(1, 1e7)
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)
ax.set_aspect('equal')

# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 1e-8 * kHz
for dim in mtrap.dimensions.values():
    ω_range_around_resonance = np.arange(dim.Ω - width, dim.Ω + width + step, step)
    jn = dim.label
    jn_string = f'{jn[0]}_{jn[1:]}'

    ax = ax_dict['P']
    vals = [mtrap.P(ω) for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$')

    ax = ax_dict['P_via_Pj']
    vals = [mtrap.P_via_Pj(ω) for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$')

    ax = ax_dict['P_via_Pj_2']
    vals = [mtrap.P_via_Pj_2(ω) for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$')

    ax = ax_dict['P_via_Pjk']
    vals = [mtrap.P_via_Pjk(ω) for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$')

    ax = ax_dict['xi P']
    vals = [mtrap.cavity.ξC(ω) * mtrap.P(ω)for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$')

    ax = ax_dict['D0']
    vals = [mtrap.D(0, ω) for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$')

    ax = ax_dict['eta D0']
    vals = [mtrap.cavity.ηm(ω) * mtrap.D(0, ω) for ω in ω_range_around_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
    ax.legend(title=r'$j$')

plt.show()


## Per-dimension components $\mathcal{P} \equiv \sum_j \mathcal{P}_j$
$j = 1,\ldots,N$

$\mathcal{P}_j \equiv 2 g_j^2 \mu_j \sum_{k \neq j} g_k^2 \mu_k \sin^2 (\phi_j - \phi_k) = g_j^2 \mu_j \mathcal{K}_j$


Notes:
- Peak of on-bead, pairing dimension is not present, since the phase difference is definitionally zero
- Imaginary part of all $k \neq j$ peaks is negative, positive for $j$-peak
- Imaginary part contributes more to the peaks than real part

In [ ]:
sl = slice(0, 2)
mosaic = [[dim.label + 'linear'] for dim in mtrap.dimensions.values()][sl]
mosaic += [[dim.label + 'log'] for dim in mtrap.dimensions.values()][sl]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.3})

for dim in list(mtrap.dimensions.values())[sl]:
    vals = [dim.P(ω) for ω in ω_range]

    ax = ax_dict[dim.label + 'linear']
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(latex_ω_units[2])
    ax.set_xlim(280, 320)
    ax.set_ylim(-0.25e20, 1e20)
    ax.vlines(dim.Ω / kHz, -0.25e20, 1e20, color=red, alpha=0.7, lw=2, ls='--', label=Ωj_str(dim))
    ax.vlines([dim.Ω / kHz for dim in dim.all_dimensions_in_other_traps], -0.25e20, 1e20, color=black, alpha=0.1, ls='--', label=r'All other $\Omega_j$')
    ax.set_title(Pj_str_1(dim))
    ax.plot(ω_range / kHz, np.real(vals), label='Re', lw=2, alpha=0.7, ls='--')
    ax.plot(ω_range / kHz, np.imag(vals), label='Im', lw=1, alpha=0.7, ls='--')
    ax.plot(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
    ax.legend(loc='center right')

    ax = ax_dict[dim.label + 'log']
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(latex_ω_units[2])
    ax.set_xlim(280, 320)
    ax.set_ylim(1e3, 1e21)
    ax.vlines(dim.Ω / kHz, 1e2, 1e22, color=red, alpha=0.7, lw=2, ls='--', label=Ωj_str(dim))
    ax.vlines([dim.Ω / kHz for dim in dim.all_dimensions_in_other_traps], 1e2, 1e22, color=black, alpha=0.1, ls='--', label=r'All other $\Omega_j$')
    ax.set_title(Pj_str_1(dim))
    ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
    ax.legend(loc='center right')

plt.show()


In [ ]:
sl = slice(0, 2)
mosaic = [[dim.label + 'linear'] for dim in mtrap.dimensions.values()][sl]
mosaic += [[dim.label + 'log'] for dim in mtrap.dimensions.values()][sl]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.3})

for dim in list(mtrap.dimensions.values())[sl]:
    vals = [dim.P_2(ω) for ω in ω_range]

    ax = ax_dict[dim.label + 'linear']
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(latex_ω_units[2])
    ax.set_xlim(280, 320)
    ax.set_ylim(-1.55e20, 1.5e20)
    ax.vlines(dim.Ω / kHz, -2e20, 2e20, color=red, alpha=0.7, lw=2, ls='--', label=Ωj_str(dim))
    ax.vlines([dim.Ω / kHz for dim in dim.all_dimensions_in_other_traps], -2e20, 2e20, color=black, alpha=0.1, ls='--', label=r'All other $\Omega_j$')
    ax.set_title(Pj_str_2(dim))
    ax.plot(ω_range / kHz, np.real(vals), label='Re', lw=2, alpha=0.7, ls='--')
    ax.plot(ω_range / kHz, np.imag(vals), label='Im', lw=1, alpha=0.7, ls='--')
    ax.plot(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
    ax.legend(loc='lower right')

    ax = ax_dict[dim.label + 'log']
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(latex_ω_units[2])
    ax.set_xlim(280, 320)
    ax.set_ylim(1e3, 1.5e21)
    ax.vlines(dim.Ω / kHz, 1e2, 2e22, color=red, alpha=0.7, lw=2, ls='--', label=Ωj_str(dim))
    ax.vlines([dim.Ω / kHz for dim in dim.all_dimensions_in_other_traps], 1e2, 2e22, color=black, alpha=0.1, ls='--', label=r'All other $\Omega_j$')
    ax.set_title(Pj_str_2(dim))
    ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
    ax.legend(loc='lower right')

plt.show()


## Complex parts of each $\mathcal{P}_j$ around $k^\mathrm{th}$ resonance

Notes:
- Circle for on-bead pairing dimension is not present, as expected
- Definition 1 gives cleanly vertically aligned circles
- Definition 2 gives circles with different orientation in pairs based on which particle they are on
- The fact they sum to the same thing is surprising?

In [ ]:
sl = slice(0, 3)

xlims, ylims = get_latex_doc_lims(12, 1e18)

# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 1e-8 * kHz
for dim_j in list(mtrap.dimensions.values())[sl]:
    jn = dim_j.label
    jn_string = f'{jn[0]}_{jn[1:]}'

    mosaic = [['1'], ['2']]
    fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
    ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

    ax = ax_dict['1']
    ax.set_aspect('equal')
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines([0], *ylims, color=black, alpha=0.1)
    ax.set_xlabel(rf'$\mathrm{{Re}}(\mathcal{{P}}_{{{jn_string}}}^{{(1)}})$' + latex_ω_units[2])
    ax.set_ylabel(rf'$\mathrm{{Im}}(\mathcal{{P}}_{{{jn_string}}}^{{(1)}})$' + latex_ω_units[2])
    ax.set_title(rf'$\mathcal{{P}}_{{{jn_string}}}^{{(1)}}$ around $k^\mathrm{{th}}$ resonance')

    ax = ax_dict['2']
    ax.set_aspect('equal')
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines([0], *ylims, color=black, alpha=0.1)
    ax.set_xlabel(rf'$\mathrm{{Re}}(\mathcal{{P}}_{{{jn_string}}}^{{(2)}})$' + latex_ω_units[2])
    ax.set_ylabel(rf'$\mathrm{{Im}}(\mathcal{{P}}_{{{jn_string}}}^{{(2)}})$' + latex_ω_units[2])
    ax.set_title(rf'$\mathcal{{P}}_{{{jn_string}}}^{{(2)}}$ around $k^\mathrm{{th}}$ resonance')

    for dim_k in dim_j.all_dimensions_in_other_traps:
        km = dim_k.label
        km_string = f'{km[0]}_{km[1:]}'
        ω_range_around_kth_resonance = np.arange(dim_k.Ω - width, dim_k.Ω + width + step, step)

        ax = ax_dict['1']
        vals = [dim_j.P(ω) for ω in ω_range_around_kth_resonance]
        ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
        ax.legend(title=r'$k$', loc='center left')

        ax = ax_dict['2']
        vals = [dim_j.P_2(ω) for ω in ω_range_around_kth_resonance]
        ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
        ax.legend(title=r'$k$', loc='center left')

plt.show()


## Per-dimension pair components $\mathcal{P} \equiv \sum_{jk} \mathcal{P}_{jk}$

$jk \in$ $N \choose 2$ $= N(N-1)/2$ unordered pairs of dimensions $j,k$

$\mathcal{P}_{jk} \equiv 4 g_j^2 \mu_j g_k^2 \mu_k \sin^2 (\phi_j - \phi_k)$

Notes:
- Only the relevant dimensions contribute

In [ ]:
# this has length N choose 2 = N(N-1)/2
all_dim_pairs = list(itertools.combinations(mtrap.dimensions.values(), 2))

# this has length N choose 2 minus number of beads if xy, minus 2 * number of beads if x,y,z
all_off_bead_dim_pairs = [
    dim_pair
    for dim_pair in list(itertools.combinations(mtrap.dimensions.values(), 2))
    if dim_pair[0].label[1:] != dim_pair[1].label[1:]
]

# for 4 beads xy, these numbers should be 8*7/2 = 28, then minus 4 = 24
# for 3 beads xy, these numbers should be 6*5/2 = 15, then minus 3 = 12
# for 2 beads xy, these numbers should be 4*3/2 = 6, then minus 2 = 4
print(len(list(all_dim_pairs)))
print(len(list(all_off_bead_dim_pairs)))

sl = slice(0, 12)
mosaic = [[dim_jn.label + dim_km.label] for (dim_jn, dim_km) in all_off_bead_dim_pairs][sl]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 1e-8 * kHz

for dim_pair in all_off_bead_dim_pairs[sl]:
    dim_jn, dim_km = dim_pair[0], dim_pair[1]
    ax = ax_dict[dim_jn.label + dim_km.label]
    ax.set_title(Pjk_str(dim_pair))
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(latex_ω_units[2])
    ax.set_xlim(280, 320)
    ax.set_ylim(1e3, 1e21)
    all_dimensions_in_other_traps = set.intersection(set(dim_jn.all_dimensions_in_other_traps), set(dim_km.all_dimensions_in_other_traps))
    ax.vlines(dim_jn.Ω / kHz, 1e2, 1e22, color=red, alpha=0.7, lw=2, ls='--', label=Ωj_str(dim_jn))
    ax.vlines(dim_km.Ω / kHz, 1e2, 1e22, color=pink, alpha=0.7, lw=2, ls='--', label=Ωj_str(dim_km))
    ax.vlines([dim.Ω / kHz for dim in all_dimensions_in_other_traps], 1e2, 1e22, color=black, alpha=0.1, ls='--', label=r'All other $\Omega_j$')
    vals = [mtrap.Pjk(dim_pair, ω) for ω in ω_range]
    ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
    ax.legend(loc='lower right')

plt.show()


## Complex parts of each $\mathcal{P}_{jk}$ around $j^\mathrm{th}$ and $k^\mathrm{th}$ resonances

Notes:
- Clean vertically aligned circles
- Negative circle is slightly larger than positive one
- % difference is dependent on difference in resonant frequencies (therefore $y_1$ vs. $x_4$ is the biggest relative difference of about $\sim$ 8%)

In [ ]:
# this has length N choose 2 = N(N-1)/2
all_dim_pairs = list(itertools.combinations(mtrap.dimensions.values(), 2))

# this has length N choose 2 minus number of beads if xy, minus 2 * number of beads if x,y,z
all_off_bead_dim_pairs = [
    dim_pair
    for dim_pair in list(itertools.combinations(mtrap.dimensions.values(), 2))
    if dim_pair[0].label[1:] != dim_pair[1].label[1:]
]

# for 4 beads xy, these numbers should be 8*7/2 = 28, then minus 4 = 24
# for 3 beads xy, these numbers should be 6*5/2 = 15, then minus 3 = 12
# for 2 beads xy, these numbers should be 4*3/2 = 6, then minus 2 = 4
print(len(list(all_dim_pairs)))
print(len(list(all_off_bead_dim_pairs)))

sl = slice(0, 12)
mosaic = [[dim_jn.label + dim_km.label] for (dim_jn, dim_km) in all_off_bead_dim_pairs][sl]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 1e-8 * kHz

count = 0
for dim_pair in all_off_bead_dim_pairs[sl]:
    dim_jn, dim_km = dim_pair[0], dim_pair[1]
    ax = ax_dict[dim_jn.label + dim_km.label]

    if count == 0:
        title_str = (
            r'$\mathcal{P}_{jk}$ around $j^\mathrm{th}$ and $k^\mathrm{th}$ resonances'
            '\n'
            f'{Pjk_str(dim_pair)}'
        )
        count = 1
    else:
        title_str = Pjk_str(dim_pair)

    ax.set_title(title_str)
    ax.set_xlabel(r'$\mathrm{Re}($' + Pjk_str(dim_pair) + '$)$' + latex_ω_units[2])
    ax.set_ylabel(r'$\mathrm{Im}($' + Pjk_str(dim_pair) + '$)$' + latex_ω_units[2])
    ax.set_aspect('equal')
    xlims, ylims = get_latex_doc_lims(2, 1e19)
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines([0], *ylims, color=black, alpha=0.1)
    for dim in [dim_jn, dim_km]:
        dim_str = f'{dim.label[0]}_{dim.label[1:]}'
        ω_range_around_resonance = np.arange(dim.Ω - width, dim.Ω + width + step, step)
        vals = [mtrap.Pjk(dim_pair, ω) for ω in ω_range_around_resonance]
        ax.plot(np.real(vals), np.imag(vals), label=rf'${{{dim_str}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim])
        ax.legend(loc='lower right')

plt.show()


In [ ]:
# the negative imaginary circle is a few % bigger than the positive imaginary circle
for jn, km in [
    ('x1', 'x2'),
    ('x1', 'y2'),
    ('x1', 'x3'),
    ('x1', 'y3'),
    ('x1', 'x4'),
    ('x1', 'y4'),
    ('y1', 'x2'),
    ('y1', 'y2'),
    ('y1', 'x3'),
    ('y1', 'y3'),
    ('y1', 'x4'),
    ('y1', 'y4'),

]:
    dim_pair = (mtrap.dimensions[jn], mtrap.dimensions[km])
    dim = mtrap.dimensions[jn]
    ω_range_around_lth_resonance = np.arange(dim.Ω - width, dim.Ω + width + step, step)
    vals_jn = [mtrap.Pjk(dim_pair, ω) for ω in ω_range_around_lth_resonance]
    print(f'{jn} real', np.min(np.real(vals_jn)), np.max(np.real(vals_jn)))
    print(f'{jn} imag', np.min(np.imag(vals_jn)), np.max(np.imag(vals_jn)))

    dim = mtrap.dimensions[km]
    ω_range_around_lth_resonance = np.arange(dim.Ω - width, dim.Ω + width + step, step)
    vals_km = [mtrap.Pjk(dim_pair, ω) for ω in ω_range_around_lth_resonance]
    print(f'{km} real', np.min(np.real(vals_km)), np.max(np.real(vals_km)))
    print(f'{km} imag', np.min(np.imag(vals_km)), np.max(np.imag(vals_km)))

    print(f'{km} vs {jn}')
    print(np.max(np.abs(np.real(vals_km))) / np.max(np.abs(np.real(vals_jn))))
    print(np.max(np.abs(np.imag(vals_km))) / np.max(np.abs(np.imag(vals_jn))))
    print()


# PART 3: Zoo of other related phase-dependent functions $\mathcal{K}_j, \mathcal{H}_j, \mathcal{U}_j$

## Different forms of $\mathcal{K}_j$
- Units: [$\mathcal{K}_j$] = $2\pi\cdot$ Hz

Definition (1)
$$ \mathcal{K}_j^{(1)} \equiv 2 \sum_{k \neq j} g_k^2 \mu_k \sin^2 (\phi_j - \phi_k) $$

Definition (2)

$$ \mathcal{K}_j^{(2)} \equiv \sum_k g_k^2 \mu_k \left( 1 - e^{2i(\phi_j - \phi_k)} \right) $$

Notes:
- Only off-bead dimensions contribute, as expected
- Why are the two definitions not equal? But they both work when summed for $\mathcal{M}$ in the end?

In [ ]:
sl = slice(0, 4)

for dim in list(mtrap.dimensions.values())[sl]:
    mosaic = [['1'], ['2']]
    fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
    ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

    j, n = dim.label[0], dim.label[1:]
    jn = f'{j}_{n}'

    vals = [dim.K(ω) for ω in ω_range]
    ax = ax_dict['1']
    ax.set_title(rf'Definition (1): ' + Kj_str(dim) +  rf'$\equiv 2 \sum_{{k \neq {{{jn}}}}} g_k^2 \mu_k \sin^2 (\phi_{{{jn}}} - \phi_k)$')
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(Kj_str(dim) + latex_ω_units[1])
    ax.set_xlim(280, 320)
    ax.set_ylim(1e-3, 1e14)
    ax.vlines(dim.Ω / kHz, 1e-4, 1e15, color=red, alpha=0.7, ls='--', label=Ωj_str(dim))
    ax.vlines([dim.Ω / kHz for dim in dim.all_dimensions_in_other_traps], 1e-4, 1e15, color=black, alpha=0.2, ls='--', label=r'All other $\Omega_j$')
    ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
    ax.legend()

    vals = [dim.K_2(ω) for ω in ω_range]
    ax = ax_dict['2']
    ax.set_title(r'Definition (2): ' + Kj_str(dim) + rf'$\equiv \sum_k g_k^2 \mu_k \left( 1 - e^{{2i(\phi_{{{jn}}} - \phi_k)}} \right)$')
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(Kj_str(dim) + latex_ω_units[1])
    ax.set_xlim(280, 320)
    ax.set_ylim(1e-3, 1e14)
    ax.vlines(dim.Ω / kHz, 1e-4, 1e15, color=red, alpha=0.7, ls='--', label=Ωj_str(dim))
    ax.vlines([dim.Ω / kHz for dim in dim.all_dimensions_in_other_traps], 1e-4, 1e15, color=black, alpha=0.2, ls='--', label=r'All other $\Omega_j$')
    ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
    ax.legend()

    plt.show()


Notes:
- $\sum_j \mathcal{K}_j$ is not equal across both definitions!
- Somehow, multiplication by $g_j^2 \mu_j$ aligns both...

In [ ]:
sl = slice(0, 4)

mosaic = [['1'], ['2'], ['3'], ['4'], ['5'], ['6']]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.6})

xlims, ylims = get_latex_doc_lims(5, 1e13)

ax = ax_dict['1']
ax.set_title(r'$\sum_j \mathcal{K}_j^{(1)}$ around $k^\mathrm{th}$ resonance')
ax.set_xlabel(r'$\mathrm{Re} \left( \sum_j \mathcal{K}_j^{(1)} \right)$' + latex_ω_units[1])
ax.set_ylabel(r'$\mathrm{Im} \left( \sum_j \mathcal{K}_j^{(1)} \right)$' + latex_ω_units[1])
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['2']
ax.set_title(r'$\sum_j \mathcal{K}_j^{(2)}$ around $k^\mathrm{th}$ resonance')
ax.set_xlabel(r'$\mathrm{Re} \left( \sum_j \mathcal{K}_j^{(2)} \right)$' + latex_ω_units[1])
ax.set_ylabel(r'$\mathrm{Im} \left( \sum_j \mathcal{K}_j^{(2)} \right)$' + latex_ω_units[1])
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

xlims, ylims = get_latex_doc_lims(8, 1e19)

ax = ax_dict['3']
ax.set_title(r'$\sum_j g_j^2 \mu_j \mathcal{K}_j^{(1)}$ around $k^\mathrm{th}$ resonance')
ax.set_xlabel(r'$\mathrm{Re} \left( \sum_j g_j^2 \mu_j \mathcal{K}_j^{(1)} \right)$' + latex_ω_units[2])
ax.set_ylabel(r'$\mathrm{Im} \left( \sum_j g_j^2 \mu_j \mathcal{K}_j^{(1)} \right)$' + latex_ω_units[2])
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['4']
ax.set_title(r'$\sum_j g_j^2 \mu_j \mathcal{K}_j^{(2)}$ around $k^\mathrm{th}$ resonance')
ax.set_xlabel(r'$\mathrm{Re} \left( \sum_j g_j^2 \mu_j \mathcal{K}_j^{(2)} \right)$' + latex_ω_units[2])
ax.set_ylabel(r'$\mathrm{Im} \left( \sum_j g_j^2 \mu_j \mathcal{K}_j^{(2)} \right)$' + latex_ω_units[2])
ax.set_aspect('equal')
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.hlines([0], *xlims, color=black, alpha=0.1)
ax.vlines([0], *ylims, color=black, alpha=0.1)

# xlims, ylims = get_latex_doc_lims(8, 1e19)

ax = ax_dict['5']
ax.set_title(r'$\sum_j \mathcal{H}_j^{(1)}$ around $k^\mathrm{th}$ resonance')
ax.set_xlabel(r'$\mathrm{Re} \left( \sum_j \mathcal{H}_j^{(1)} \right)$')
ax.set_ylabel(r'$\mathrm{Im} \left( \sum_j \mathcal{H}_j^{(1)} \right)$')
ax.set_aspect('equal')
# ax.set_xlim(*xlims)
# ax.set_ylim(*ylims)
# ax.hlines([0], *xlims, color=black, alpha=0.1)
# ax.vlines([0], *ylims, color=black, alpha=0.1)

ax = ax_dict['6']
ax.set_title(r'$\sum_j \mathcal{H}_j^{(2)}$ around $k^\mathrm{th}$ resonance')
ax.set_xlabel(r'$\mathrm{Re} \left( \sum_j \mathcal{H}_j^{(2)} \right)$')
ax.set_ylabel(r'$\mathrm{Im} \left( \sum_j \mathcal{H}_j^{(2)} \right)$')
ax.set_aspect('equal')
# ax.set_xlim(*xlims)
# ax.set_ylim(*ylims)
# ax.hlines([0], *xlims, color=black, alpha=0.1)
# ax.vlines([0], *ylims, color=black, alpha=0.1)


# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 2e-8 * kHz

for dim_k in list(mtrap.dimensions.values()):
    km = dim_k.label
    km_string = f'{km[0]}_{km[1:]}'
    ω_range_around_kth_resonance = np.arange(dim_k.Ω - width, dim_k.Ω + width + step, step)

    ax = ax_dict['1']
    vals = [np.sum([dim_j.K(ω) for dim_j in mtrap.dimensions.values()]) for ω in ω_range_around_kth_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
    ax.legend(title=r'$k$', loc='center left')

    ax = ax_dict['2']
    vals = [np.sum([dim_j.K_2(ω) for dim_j in mtrap.dimensions.values()]) for ω in ω_range_around_kth_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
    ax.legend(title=r'$k$', loc='center left')

    ax = ax_dict['3']
    vals = [np.sum([dim_j.g**2 * dim_j.μm(ω) * dim_j.K(ω) for dim_j in mtrap.dimensions.values()]) for ω in ω_range_around_kth_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
    ax.legend(title=r'$k$', loc='center left')

    ax = ax_dict['4']
    vals = [np.sum([dim_j.g**2 * dim_j.μm(ω) * dim_j.K_2(ω) for dim_j in mtrap.dimensions.values()]) for ω in ω_range_around_kth_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
    ax.legend(title=r'$k$', loc='center left')

    option = 'H'
    ax = ax_dict['5']
    vals = [np.sum([dim_j.N(option, ω) for dim_j in mtrap.dimensions.values()]) for ω in ω_range_around_kth_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
    # ax.legend(title=r'$k$', loc='center left')

    ax = ax_dict['6']
    vals = [np.sum([dim_j.N_2(option, ω) for dim_j in mtrap.dimensions.values()]) for ω in ω_range_around_kth_resonance]
    ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
    # ax.legend(title=r'$k$', loc='center left')

plt.show()


## Complex parts of $\mathcal{K}_j$ around $k^\mathrm{th}$ resonance

Notes:
- Definition 1 as cleanly horizontally aligned cirlces
- Definition 2 each circle is rotated, paired by bead

In [ ]:
sl = slice(0, 4)

xlims, ylims = get_latex_doc_lims(1, 1e13)

# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 1e-8 * kHz
for dim_j in list(mtrap.dimensions.values())[sl]:
    jn = dim_j.label
    jn_string = f'{jn[0]}_{jn[1:]}'

    mosaic = [['1'], ['2']]
    fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
    ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

    ax = ax_dict['1']
    ax.set_title(rf'$\mathcal{{K}}_{{{jn_string}}}^{{(1)}}$ around $k^\mathrm{{th}}$ resonance')
    ax.set_xlabel(rf'$\mathrm{{Re}}(\mathcal{{K}}_{{{jn_string}}}^{{(1)}})$' + latex_ω_units[1])
    ax.set_ylabel(rf'$\mathrm{{Im}}(\mathcal{{K}}_{{{jn_string}}}^{{(1)}})$' + latex_ω_units[1])
    ax.set_aspect('equal')
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines([0], *ylims, color=black, alpha=0.1)

    ax = ax_dict['2']
    ax.set_title(rf'$\mathcal{{K}}_{{{jn_string}}}^{{(2)}}$ around $k^\mathrm{{th}}$ resonance')
    ax.set_xlabel(rf'$\mathrm{{Re}}(\mathcal{{K}}_{{{jn_string}}}^{{(2)}})$' + latex_ω_units[1])
    ax.set_ylabel(rf'$\mathrm{{Im}}(\mathcal{{K}}_{{{jn_string}}}^{{(2)}})$' + latex_ω_units[1])
    ax.set_aspect('equal')
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines([0], *ylims, color=black, alpha=0.1)

    for dim_k in dim_j.all_dimensions_in_other_traps:
        km = dim_k.label
        km_string = f'{km[0]}_{km[1:]}'
        ω_range_around_kth_resonance = np.arange(dim_k.Ω - width, dim_k.Ω + width + step, step)

        ax = ax_dict['1']
        vals = [dim_j.K(ω) for ω in ω_range_around_kth_resonance]
        ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
        ax.legend(title=r'$k$', loc='center left')

        ax = ax_dict['2']
        vals = [dim_j.K_2(ω) for ω in ω_range_around_kth_resonance]
        ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
        ax.legend(title=r'$k$', loc='center left')

plt.show()


## Different forms of $\mathcal{H}_j, \mathcal{H}_j^\dagger$

- Units: [$\mathcal{H}_j$] = 1

$$ \mathcal{H}_j \equiv -\chi_\mathrm{c}^*(-\omega) \mathcal{K}_j $$
$$ \mathcal{H}_j^\dagger \equiv \chi_\mathrm{c}(\omega) \mathcal{K}_j $$

with either definition of $\mathcal{K}_j$ i.e.

Definition (1)
$$ \mathcal{H}_j^{(1)} \equiv -2 \chi_\mathrm{c}^*(-\omega) \sum_{k \neq j} g_k^2 \mu_k \sin^2 (\phi_j - \phi_k) $$
$$ [\mathcal{H}_j^{(1)}]^\dagger \equiv 2 \chi_\mathrm{c}(\omega) \sum_{k \neq j} g_k^2 \mu_k \sin^2 (\phi_j - \phi_k) $$

Definition (2)
$$ \mathcal{H}_j^{(2)} \equiv \chi_\mathrm{c}^*(-\omega) \sum_k g_k^2 \mu_k \left(e^{2i(\phi_j - \phi_k)} - 1 \right) $$
$$ [\mathcal{H}_j^{(2)}]^\dagger \equiv \chi_\mathrm{c}(\omega) \sum_k g_k^2 \mu_k \left( 1 - e^{-2i(\phi_j - \phi_k)} \right) $$


In [ ]:
sl = slice(0, 3)

for dim in list(mtrap.dimensions.values())[sl]:
    mosaic = [['1'], ['2']]
    fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
    ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

    j, n = dim.label[0], dim.label[1:]
    jn = f'{j}_{n}'

    vals = [dim.H(ω) for ω in ω_range]
    ax = ax_dict['1']
    ax.set_title(rf'Definition (1): ' + Hj_str(dim) +  rf'$\equiv -2 \chi_\mathrm{{c}}^*(-\omega) \sum_{{k \neq {{{jn}}}}} g_k^2 \mu_k \sin^2 (\phi_{{{jn}}} - \phi_k)$')
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(Hj_str(dim))
    ax.set_xlim(280, 320)
    ax.set_ylim(1e-4, 1e8)
    ax.vlines(dim.Ω / kHz, 1e-4, 1e15, color=red, alpha=0.7, ls='--', label=Ωj_str(dim))
    ax.vlines([dim.Ω / kHz for dim in dim.all_dimensions_in_other_traps], 1e-4, 1e15, color=black, alpha=0.2, ls='--', label=r'All other $\Omega_j$')
    ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
    ax.legend()

    vals = [dim.H_2(ω) for ω in ω_range]
    ax = ax_dict['2']
    ax.set_title(r'Definition (2): ' + Hj_str(dim) + rf'$\equiv \chi_\mathrm{{c}}^*(-\omega) \sum_k g_k^2 \mu_k \left( 1 - e^{{2i(\phi_{{{jn}}} - \phi_k)}} \right)$')
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(Hj_str(dim))
    ax.set_xlim(280, 320)
    ax.set_ylim(1e-4, 1e8)
    ax.vlines(dim.Ω / kHz, 1e-4, 1e15, color=red, alpha=0.7, ls='--', label=Ωj_str(dim))
    ax.vlines([dim.Ω / kHz for dim in dim.all_dimensions_in_other_traps], 1e-4, 1e15, color=black, alpha=0.2, ls='--', label=r'All other $\Omega_j$')
    ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
    ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
    ax.legend()

    plt.show()


## Complex parts of $\mathcal{H}_j$, $\mathcal{H}_j^\dagger$ around $k^\mathrm{th}$ resonance

Notes:
- Multiplication by $\chi_\mathrm{c}$ rotates all circles upwards

In [ ]:
sl = slice(0, 4)

xlims, ylims = get_latex_doc_lims(7, 1e6)

# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 1e-8 * kHz
for dim_j in list(mtrap.dimensions.values())[sl]:
    jn = dim_j.label
    jn_string = f'{jn[0]}_{jn[1:]}'

    mosaic = [['1'], ['2']]
    fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
    ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

    ax = ax_dict['1']
    ax.set_title(rf'$\mathcal{{H}}_{{{jn_string}}}^{{(1)}}$ around $k^\mathrm{{th}}$ resonance')
    ax.set_xlabel(rf'$\mathrm{{Re}}(\mathcal{{H}}_{{{jn_string}}}^{{(1)}})$')
    ax.set_ylabel(rf'$\mathrm{{Im}}(\mathcal{{H}}_{{{jn_string}}}^{{(1)}})$')
    ax.set_aspect('equal')
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines([0], *ylims, color=black, alpha=0.1)

    ax = ax_dict['2']
    ax.set_title(rf'$\mathcal{{H}}_{{{jn_string}}}^{{(2)}}$ around $k^\mathrm{{th}}$ resonance')
    ax.set_xlabel(rf'$\mathrm{{Re}}(\mathcal{{H}}_{{{jn_string}}}^{{(2)}})$')
    ax.set_ylabel(rf'$\mathrm{{Im}}(\mathcal{{H}}_{{{jn_string}}}^{{(2)}})$')
    ax.set_aspect('equal')
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines([0], *ylims, color=black, alpha=0.1)

    for dim_k in dim_j.all_dimensions_in_other_traps:
        km = dim_k.label
        km_string = f'{km[0]}_{km[1:]}'
        ω_range_around_kth_resonance = np.arange(-dim_k.Ω - width, -dim_k.Ω + width + step, step)

        ax = ax_dict['1']
        vals = [dim_j.H(ω) for ω in ω_range_around_kth_resonance]
        ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
        ax.legend(title=r'$k$', loc='center left')

        ax = ax_dict['2']
        vals = [dim_j.H_2(ω) for ω in ω_range_around_kth_resonance]
        ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
        ax.legend(title=r'$k$', loc='center left')

plt.show()


In [ ]:
sl = slice(0, 4)

xlims, ylims = get_latex_doc_lims(7, 1e6)

# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 1e-8 * kHz
for dim_j in list(mtrap.dimensions.values())[sl]:
    jn = dim_j.label
    jn_string = f'{jn[0]}_{jn[1:]}'

    mosaic = [['1'], ['2']]
    fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
    ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

    ax = ax_dict['1']
    ax.set_title(rf'$[\mathcal{{H}}_{{{jn_string}}}^{{(1)}}]^\dagger$ around $k^\mathrm{{th}}$ resonance')
    ax.set_xlabel(rf'$\mathrm{{Re}}([\mathcal{{H}}_{{{jn_string}}}^{{(1)}}]^\dagger)$')
    ax.set_ylabel(rf'$\mathrm{{Im}}([\mathcal{{H}}_{{{jn_string}}}^{{(1)}}]^\dagger)$')
    ax.set_aspect('equal')
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines([0], *ylims, color=black, alpha=0.1)

    ax = ax_dict['2']
    ax.set_title(rf'$[\mathcal{{H}}_{{{jn_string}}}^{{(2)}}]^\dagger$ around $k^\mathrm{{th}}$ resonance')
    ax.set_xlabel(rf'$\mathrm{{Re}}([\mathcal{{H}}_{{{jn_string}}}^{{(2)}}]^\dagger)$')
    ax.set_ylabel(rf'$\mathrm{{Im}}([\mathcal{{H}}_{{{jn_string}}}^{{(2)}}]^\dagger)$')
    ax.set_aspect('equal')
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines([0], *ylims, color=black, alpha=0.1)

    for dim_k in dim_j.all_dimensions_in_other_traps:
        km = dim_k.label
        km_string = f'{km[0]}_{km[1:]}'
        ω_range_around_kth_resonance = np.arange(dim_k.Ω - width, dim_k.Ω + width + step, step)

        ax = ax_dict['1']
        vals = [dim_j.H_dag(ω) for ω in ω_range_around_kth_resonance]
        ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
        ax.legend(title=r'$k$', loc='center left')

        ax = ax_dict['2']
        vals = [dim_j.H_dag_2(ω) for ω in ω_range_around_kth_resonance]
        ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
        ax.legend(title=r'$k$', loc='center left')

plt.show()


## Different forms of $\mathcal{N}_j$

$$
    \mathcal{M} \equiv 1 + \sum_j g_j^2 \mu_j \mathcal{N}_j
$$

If all $\phi_j$ are equal, $\mathcal{N}_j = \eta_\mathrm{c}$

Definition (i)
$$ \mathcal{N}_j \equiv \eta_{\mathrm{c}} - \xi_{\mathrm{c}} \mathcal{K}_j $$

Definition (ii)
$$ \mathcal{N}_j \equiv \eta_{\mathrm{c}} + \chi_{\mathrm{c}}(\omega) \mathcal{H}_j $$

Definition (iii)
$$ \mathcal{N}_j \equiv \chi_{\mathrm{c}}(\omega) \mathcal{U}_j - \chi_{\mathrm{c}}^*(-\omega) $$

Definition (iv)
$$ \mathcal{N}_j \equiv \eta_{\mathrm{c}} - \chi_{\mathrm{c}}^*(-\omega) \mathcal{H}_j^{\dagger} $$

Definition (v)
$$ \mathcal{N}_j \equiv \chi_{\mathrm{c}}(\omega) - \chi_{\mathrm{c}}^*(-\omega)\mathcal{U}_j^{\dagger} $$


In [ ]:
sl = slice(0, 1)

options = option_to_latex_Nj

for dim in list(mtrap.dimensions.values())[sl]:
    for option, latex in options.items():
        mosaic = [['1'], ['2']]
        fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
        ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

        j, n = dim.label[0], dim.label[1:]
        jn = f'{j}_{n}'

        def_n = option_defs[option]

        vals = [dim.N(option, ω) for ω in ω_range]
        ax = ax_dict['1']
        ax.set_title(rf'Definition (1{def_n}: $\mathcal{{N}}_j \equiv$'  +  latex)
        ax.set_xlabel('Frequency [kHz]')
        ax.set_ylabel(Nj_str(dim) + latex_ω_units[-1])
        ax.set_xlim(280, 320)
        ax.set_ylim(1e-12, 1e2)
        ax.vlines(dim.Ω / kHz, 1e-12, 1e2, color=red, alpha=0.7, ls='--', label=Ωj_str(dim))
        ax.vlines([dim.Ω / kHz for dim in dim.all_dimensions_in_other_traps], 1e-12, 1e2, color=black, alpha=0.2, ls='--', label=r'All other $\Omega_j$')
        ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
        ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
        ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
        ax.legend()

        vals = [dim.N_2(option, ω) for ω in ω_range]
        ax = ax_dict['2']
        ax.set_title(rf'Definition (2{def_n}: $\mathcal{{N}}_j \equiv$'  +  latex)
        ax.set_xlabel('Frequency [kHz]')
        ax.set_ylabel(Nj_str(dim) + latex_ω_units[-1])
        ax.set_xlim(280, 320)
        ax.set_ylim(1e-12, 1e2)
        ax.vlines(dim.Ω / kHz, 1e-12, 1e2, color=red, alpha=0.7, ls='--', label=Ωj_str(dim))
        ax.vlines([dim.Ω / kHz for dim in dim.all_dimensions_in_other_traps], 1e-12, 1e2, color=black, alpha=0.2, ls='--', label=r'All other $\Omega_j$')
        ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), label='|Re|', lw=2, alpha=0.7, ls='--')
        ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), label='|Im|', lw=1, alpha=0.7, ls='--')
        ax.semilogy(ω_range / kHz, np.abs(vals), label='Abs', lw=4, alpha=0.4)
        ax.legend()

        plt.show()


## Complex parts of $\mathcal{N}_j$ around $k^\mathrm{th}$ resonance

Notes:
- Expressions with Definition 1 are all equal, all being aligned circles of varying size
- Expressions with Definition 2 with $\mathcal{K}_j$, $\mathcal{H}_j$, $\mathcal{U}_j$ are equal, with circles paired by bead
- Expressions with Definition 2 with $\mathcal{H}_j^\dagger$, $\mathcal{U}_j^\dagger$ are equal, with circles paired by bead
- If $\mathcal{H}_j^\dagger$ is defined without reversing the $i \to -i$ in the exponent, then it becomes equal to the others
- ALL are equal when summed...


In [ ]:
sl = slice(0, 1)

xlims, ylims = get_latex_doc_lims(1.5, 1e0)

# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 1e-8 * kHz

for dim_j in list(mtrap.dimensions.values())[sl]:
    for option, latex in option_to_latex_Nj.items():
        jn = dim_j.label
        jn_string = f'{jn[0]}_{jn[1:]}'

        mosaic = [['1'], ['2']]
        fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
        ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

        def_n = option_defs[option]

        ax = ax_dict['1']
        ax.set_title(rf'Definition (1{def_n}: $\mathcal{{N}}_j \equiv$'  +  latex + '\n' + rf'$\mathcal{{N}}_{{{jn_string}}}^{{(1)}}$ around $k^\mathrm{{th}}$ resonance')
        ax.set_xlabel(rf'$\mathrm{{Re}}(\mathcal{{N}}_{{{jn_string}}}^{{(1)}})$' + latex_ω_units[-1])
        ax.set_ylabel(rf'$\mathrm{{Im}}(\mathcal{{N}}_{{{jn_string}}}^{{(1)}})$' + latex_ω_units[-1])
        ax.set_aspect('equal')
        ax.set_xlim(*xlims)
        ax.set_ylim(*ylims)
        ax.hlines([0], *xlims, color=black, alpha=0.1)
        ax.vlines([0], *ylims, color=black, alpha=0.1)

        ax = ax_dict['2']
        ax.set_title(rf'Definition (2{def_n}: $\mathcal{{N}}_j \equiv$'  +  latex + '\n' + rf'$\mathcal{{N}}_{{{jn_string}}}^{{(2)}}$ around $k^\mathrm{{th}}$ resonance')
        ax.set_xlabel(rf'$\mathrm{{Re}}(\mathcal{{N}}_{{{jn_string}}}^{{(2)}})$' + latex_ω_units[-1])
        ax.set_ylabel(rf'$\mathrm{{Im}}(\mathcal{{N}}_{{{jn_string}}}^{{(2)}})$' + latex_ω_units[-1])
        ax.set_aspect('equal')
        ax.set_xlim(*xlims)
        ax.set_ylim(*ylims)
        ax.hlines([0], *xlims, color=black, alpha=0.1)
        ax.vlines([0], *ylims, color=black, alpha=0.1)

        for dim_k in dim_j.all_dimensions_in_other_traps:
            km = dim_k.label
            km_string = f'{km[0]}_{km[1:]}'
            ω_range_around_kth_resonance = np.arange(dim_k.Ω - width, dim_k.Ω + width + step, step)

            ax = ax_dict['1']
            vals = [dim_j.N(option, ω) for ω in ω_range_around_kth_resonance]
            ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
            ax.legend(title=r'$k$', loc='center right')

            ax = ax_dict['2']
            vals = [dim_j.N_2(option, ω) for ω in ω_range_around_kth_resonance]
            ax.plot(np.real(vals), np.imag(vals), label=rf'${{{km_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_k])
            ax.legend(title=r'$k$', loc='center right')

    plt.show()


In [ ]:
options = option_to_latex_Nj

mosaic = [[option] for option in options.keys()]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

dim_j = mtrap.dimensions['y2']

for option, latex in options.items():
    ax = ax_dict[option]
    ax.set_title(r'$\mathcal{N}_j \equiv$' + latex + '\n' + rf'$j = {dim_j.label[0]}_{{{dim_j.label[1:]}}}$')
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(r'$|\mathrm{Re}(\mathcal{N}_j)|$')
    ax.set_xlim(280, 320)
    ax.set_ylim(1e-10, 1e2)
    ax.vlines(dim_j.Ω / kHz, 1e-11, 1e3, color=red, alpha=0.7, ls='--', label=Ωj_str(dim_j))
    ax.vlines([dim.Ω / kHz for dim in dim.all_dimensions_in_other_traps], 1e-11, 1e4, color=black, alpha=0.1, ls='--', label=r'All other $\Omega_j$')
    ax.semilogy(ω_range / kHz, [np.abs(np.real(dim_j.N(option, ω))) for ω in ω_range], lw=3, alpha=0.5, ls='--', label=r'$\mathcal{N}_j^{(1)}$')
    ax.semilogy(ω_range / kHz, [np.abs(np.real(dim_j.N_2(option, ω))) for ω in ω_range], lw=2, alpha=0.5, label=r'$\mathcal{N}_j^{(2)}$')
    ax.legend()

plt.show()


# PART 4: All forms of $\mathcal{M}$ are equal!

In [ ]:
xlims, ylims = get_latex_doc_lims(1.5, 1e7)

# Γj ~ 1e-6 * kHz at pressure of 1e-6 mbar
width = 1e-5 * kHz
step = 2e-8 * kHz

for option, latex in option_to_latex_M.items():
    mosaic = [['1'], ['2']]
    fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
    ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.7})

    def_n = option_defs[option]

    ax = ax_dict['1']
    ax.set_title(rf'Definition (1{def_n}: $\mathcal{{M}} \equiv$'  +  latex + '\n' + r'around $k^\mathrm{th}$ resonance')
    ax.set_xlabel(rf'$\mathrm{{Re}}(\mathcal{{M}}^{{(1)}})$')
    ax.set_ylabel(rf'$\mathrm{{Im}}(\mathcal{{M}}^{{(1)}})$')
    ax.set_aspect('equal')
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines([0], *ylims, color=black, alpha=0.1)

    ax = ax_dict['2']
    ax.set_title(rf'Definition (2{def_n}: $\mathcal{{M}} \equiv$'  +  latex + '\n' + r'around $k^\mathrm{th}$ resonance')
    ax.set_xlabel(rf'$\mathrm{{Re}}(\mathcal{{M}}^{{(2)}})$')
    ax.set_ylabel(rf'$\mathrm{{Im}}(\mathcal{{M}}^{{(2)}})$')
    ax.set_aspect('equal')
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.hlines([0], *xlims, color=black, alpha=0.1)
    ax.vlines([0], *ylims, color=black, alpha=0.1)

    for dim_j in mtrap.dimensions.values():
        jn = dim_j.label
        jn_string = f'{jn[0]}_{jn[1:]}'
        ω_range_around_jth_resonance = np.arange(dim_j.Ω - width, dim_j.Ω + width + step, step)

        ax = ax_dict['1']
        vals = [mtrap.M_norm_N(option, ω) for ω in ω_range_around_jth_resonance]
        ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_j])
        ax.legend(title=r'$j$', loc='center right')

        ax = ax_dict['2']
        vals = [mtrap.M_norm_N_2(option, ω) for ω in ω_range_around_jth_resonance]
        ax.plot(np.real(vals), np.imag(vals), label=rf'${{{jn_string}}}$', lw=1, alpha=0.7, color=dim_to_colour[dim_j])
        ax.legend(title=r'$j$', loc='center right')

    plt.show()


In [ ]:
options = option_to_latex_M

mosaic = [['Re' + option] for option in options.keys()]
mosaic += [['Im' + option] for option in options.keys()]
# mosaic += [['Abs' + option] for option in options.keys()]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

func = np.real
dim_j = mtrap.dimensions['x1']

vals = [mtrap.M_norm_N(option, ω) for ω in ω_range]
vals_2 = [mtrap.M_norm_N_2(option, ω) for ω in ω_range]

for option, latex in options.items():
    ax = ax_dict['Re' + option]
    ax.set_title(r'$\mathcal{M} \equiv$' + latex)
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(r'$|\mathrm{Re}(\mathcal{M})|$')
    ax.set_xlim(280, 320)
    # ax.set_ylim(1e-9, 1e2)
    ax.semilogy(ω_range / kHz, np.abs(np.real(vals)), lw=3, alpha=0.5, label='(1)')
    ax.semilogy(ω_range / kHz, np.abs(np.real(vals_2)), lw=2, alpha=0.5, ls='--', label='(2)')
    ax.legend(title='Definition')

    ax = ax_dict['Im' + option]
    ax.set_title(r'$\mathcal{M} \equiv$' + latex)
    ax.set_xlabel('Frequency [kHz]')
    ax.set_ylabel(r'$|\mathrm{Im}(\mathcal{M})|$')
    ax.set_xlim(280, 320)
    # ax.set_ylim(1e-9, 1e2)
    ax.semilogy(ω_range / kHz, np.abs(np.imag(vals)), lw=3, alpha=0.5, label='(1)')
    ax.semilogy(ω_range / kHz, np.abs(np.imag(vals_2)), lw=2, alpha=0.5, ls='--', label='(2)')
    ax.legend(title='Definition')

plt.show()


In [ ]:
mosaic = [
    ['Re', 'Im'],
    ['Abs', 'Abs'],
]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.8})

fig.suptitle(
    r'$\mathcal{M} \equiv$'
    f'{option_to_latex_M["K"]}'
    ' vs. '
    f'{option_to_latex_M["H"]}'
    ' vs. '
    f'{option_to_latex_M["H_dag"]}'
)

vals = [mtrap.M_norm(ω) for ω in ω_range]
vals_2 = [mtrap.M_norm_2(ω) for ω in ω_range]
vals_3 = [mtrap.M_norm_3(ω) for ω in ω_range]
vals_4 = [mtrap.M_norm_4(ω) for ω in ω_range]

func = np.real
ax = ax_dict['Re']
ax.set_title('Real part')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_2)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_3)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_4)), lw=0.5, alpha=0.7, ls='--')

func = np.imag
ax = ax_dict['Im']
ax.set_title('Imaginary part')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_2)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_3)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_4)), lw=0.5, alpha=0.7, ls='--')

func = np.abs
ax = ax_dict['Abs']
ax.set_title('Magnitude')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_2)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_3)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_4)), lw=0.5, alpha=0.7, ls='--')

plt.show()


In [ ]:
mosaic = [
    ['Re', 'Im'],
    ['Abs', 'Abs'],
]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.8})

fig.suptitle(
    r'$\mathcal{M} \equiv$'
    f'{option_to_latex_M["U"]}'
    ' vs. '
    f'{option_to_latex_M["U_dag"]}'
    ' vs. '
    r'$1 + 2i\xi_\mathrm{c} \sum_j g_j^2 \mu_j \left[ \Delta + i \mathcal{K}_j / 2 \right]$'
)

vals = [mtrap.M_norm(ω) for ω in ω_range]
vals_5 = [mtrap.M_norm_5(ω) for ω in ω_range]
vals_6 = [mtrap.M_norm_6(ω) for ω in ω_range]
vals_7 = [mtrap.M_norm_7(ω) for ω in ω_range]

func = np.real
ax = ax_dict['Re']
ax.set_title('Real part')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_5)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_6)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_7)), lw=0.5, alpha=0.7, ls='--')

func = np.imag
ax = ax_dict['Im']
ax.set_title('Imaginary part')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_5)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_6)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_7)), lw=0.5, alpha=0.7, ls='--')

func = np.abs
ax = ax_dict['Abs']
ax.set_title('Magnitude')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_5)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_6)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_7)), lw=0.5, alpha=0.7, ls='--')

plt.show()


In [ ]:
mosaic = [
    ['Re', 'Im'],
    ['Abs', 'Abs'],
]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.8})

fig.suptitle(
    r'$\mathcal{M} \equiv$'
    f'{option_to_latex_M["K"]}'
    ' vs. '
    f'{option_to_latex_M["H"]}'
    ' vs. '
    f'{option_to_latex_M["H_dag"]}'
)

vals = [mtrap.M_norm(ω) for ω in ω_range]
vals_2 = [mtrap.M_norm_2_2(ω) for ω in ω_range]
vals_3 = [mtrap.M_norm_3_2(ω) for ω in ω_range]
vals_4 = [mtrap.M_norm_4_2(ω) for ω in ω_range]

func = np.real
ax = ax_dict['Re']
ax.set_title('Real part')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_2)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_3)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_4)), lw=0.5, alpha=0.7, ls='--')

func = np.imag
ax = ax_dict['Im']
ax.set_title('Imaginary part')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_2)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_3)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_4)), lw=0.5, alpha=0.7, ls='--')

func = np.abs
ax = ax_dict['Abs']
ax.set_title('Magnitude')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_2)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_3)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_4)), lw=0.5, alpha=0.7, ls='--')

plt.show()


In [ ]:
mosaic = [
    ['Re', 'Im'],
    ['Abs', 'Abs'],
]
fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5 * len(mosaic)), dpi=150)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.8})

fig.suptitle(
    r'$\mathcal{M} \equiv$'
    f'{option_to_latex_M["U"]}'
    ' vs. '
    f'{option_to_latex_M["U_dag"]}'
    ' vs. '
    r'$1 + 2i\xi_\mathrm{c} \sum_j g_j^2 \mu_j \left[ \Delta + i \mathcal{K}_j / 2 \right]$'
)

vals = [mtrap.M_norm(ω) for ω in ω_range]
vals_5 = [mtrap.M_norm_5_2(ω) for ω in ω_range]
vals_6 = [mtrap.M_norm_6_2(ω) for ω in ω_range]
vals_7 = [mtrap.M_norm_7_2(ω) for ω in ω_range]

func = np.real
ax = ax_dict['Re']
ax.set_title('Real part')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_5)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_6)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_7)), lw=0.5, alpha=0.7, ls='--')

func = np.imag
ax = ax_dict['Im']
ax.set_title('Imaginary part')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_5)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_6)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_7)), lw=0.5, alpha=0.7, ls='--')

func = np.abs
ax = ax_dict['Abs']
ax.set_title('Magnitude')
ax.set_xlabel('Frequency (kHz)')
ax.semilogy(ω_range / kHz, np.abs(func(vals)), lw=4, alpha=0.4)
ax.semilogy(ω_range / kHz, np.abs(func(vals_5)), lw=2, alpha=0.5, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_6)), lw=1, alpha=0.6, ls='--')
ax.semilogy(ω_range / kHz, np.abs(func(vals_7)), lw=0.5, alpha=0.7, ls='--')

plt.show()


# END